# ACORN — Prediction Analysis

Evaluation of the **ACORN** (Attention COnvolutional Residual Network) field-aligned
current model against AMPERE observations, alongside two baselines.

## What is compared

| | |
|---|---|
| **AMPERE** | Observed FAC. The target, and ground truth for every metric here. |
| **ACORN Sci** | Science model — full input set including SuperMAG indices. |
| **ACORN Op** | Operational model — real-time-available inputs only. |
| **Kunduri Sci** | Science model — full input set including SuperMAG indices. |
| **Kunduri Op** | Operational model — real-time-available inputs only. |
| **Weimer** | Empirical model from CCMC. May 2023 storm window only. |

## How the analysis is organised

1. **Integrated current** — total current through the polar cap, the headline scalar.
2. **Polar maps** — spatial structure at individual timesteps and averaged.
3. **Correlation and RMSE** — per-region skill across the whole test set.
4. **Regional metrics** — HSS, normalised RMSE and AUC-PR by MLAT band and MLT sector.
5. **Temporal heatmaps** — keograms showing the R1/R2 boundary moving through a storm.
6. **Conditional analysis** — whether the model responds correctly to IMF orientation.

## Conventions

- Grid is **50 MLAT × 24 MLT**, row 0 at the pole (~89°) down to row 49 (~40°).
- Positive FAC is **upward**, out of the ionosphere.
- All polar plots go through `plotting_utils` via `polar_setup()`: midnight at the
  bottom, MLT counter-clockwise, radial axis labelled in MLAT.
- Set `LAT_MASK_50 = True` in the config cell to restrict to MLAT ≥ 50° for
  like-for-like comparison against Weimer's coverage.

## Requirements

Results pickles in `outputs/`, the AMPERE cache, and the Weimer CSV. The
gap-filling cell can generate missing predictions but is slow and needs network
access, so it is left commented by default.

## Imports

`utils`, `plotting_utils`, `inference` and `model_classes` are the repository
modules — run this notebook from the repository root so they resolve.

In [ ]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as colors
import seaborn as sns

import tqdm
from scipy.signal import find_peaks
from sklearn.metrics import mean_squared_error as MSE
from geospacepy.spherical_geometry import grid_surface_integral

import utils
import plotting_utils as pu
from inference import FACInference, TFACInference
from model_classes import ACORN

pd.options.mode.chained_assignment = None
warnings.filterwarnings('ignore')

## Configuration

Both models are loaded from the single `config.json`. Results paths are set
explicitly rather than through `utils.results_file()`.

In [ ]:
# One config.json now holds both models; load_config merges the shared
# block with the per-model one.
SCI_CONFIG = utils.load_config('sci')
OP_CONFIG  = utils.load_config('op')

# Tag appended to all plot directories and filenames to distinguish runs.
RUN_TAG = 'final'

# When True, ACORN/AMPERE grids are trimmed to MLAT >= 50 deg (rows 0-39)
# to match the Weimer coverage. Set False to use the full 40-89 deg grid.
LAT_MASK_50 = False

may_start = pd.to_datetime('2023-05-05 00:00:00')
may_end   = pd.to_datetime('2023-05-09 00:00:00')

# Results files. These are the runs being evaluated -- set explicitly
# rather than via utils.results_file() so a specific historical run can be
# compared. Switch to utils.results_file(SCI_CONFIG) for the current run.
# SCI_RESULTS = 'outputs/results_sci.pkl'
# OP_RESULTS  = 'outputs/results_op.pkl'
SCI_RESULTS = 'outputs/acorn_sci_results.pkl'
OP_RESULTS  = 'outputs/acorn_op_results.pkl'
BK_SCI_RESULTS = 'outputs/FAC_BK_sci_results.pkl'
BK_OP_RESULTS  = 'outputs/FAC_BK_op_results.pkl'

print(f'Sci inputs: {len(SCI_CONFIG["input_params"])}  {SCI_CONFIG["input_params"]}')
print(f'Op  inputs: {len(OP_CONFIG["input_params"])}  {OP_CONFIG["input_params"]}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Polar axis helper
# ═══════════════════════════════════════════════════════════════════════════
# plotting_utils.polar_axis owns the project convention -- midnight at the
# bottom, MLT counter-clockwise, radial axis in colatitude labelled in MLAT.
# Every polar plot below routes through it and then overrides only what that
# particular figure needs (tick label text, font sizes, grid alpha, rlim).
#

def polar_setup(ax, n_rad, rad_ticks=None, rad_labels=None,
                mlt_labels=('', '3', '', '9', '', '15', '', '21'),
                mlt_fontsize=10, rad_fontsize=10, rad_color=None,
                grid_alpha=0.4, grid_zorder=None, ylim=None):
    """
    Apply the project polar convention, then this figure's overrides.

    Args:
        ax: existing polar axis.
        n_rad: number of radial (MLAT) rows -- 40 when LAT_MASK_50, else 50.
        rad_ticks, rad_labels: radial ticks and their text. Defaults to
            whatever polar_axis chose.
        mlt_labels: 8 labels at 3-hour spacing. Blank entries suppress
            alternating labels.
        ylim: outer radial limit. Defaults to n_rad.
    """
    # Convention first: theta zero location, direction, MLAT labelling.
    pu.polar_axis(ax=ax, n_mlat=n_rad, grid=False)

    # Then this figure's overrides.
    theta_ticks = np.linspace(0, 2 * np.pi, 8, endpoint=False)
    ax.set_xticks(theta_ticks)
    ax.set_xticklabels(list(mlt_labels), fontsize=mlt_fontsize)

    if rad_ticks is not None:
        ax.set_yticks(rad_ticks)
    if rad_labels is not None:
        kw = {'fontsize': rad_fontsize}
        if rad_color is not None:
            kw['color'] = rad_color
        ax.set_yticklabels(rad_labels, **kw)

    ax.set_ylim(0, n_rad if ylim is None else ylim)

    grid_kw = {'alpha': grid_alpha}
    if grid_zorder is not None:
        grid_kw['zorder'] = grid_zorder
    ax.grid(**grid_kw)

    return ax


def mlat_labels(rad_ticks, blank_pole=True):
    """
    MLAT text for a set of colatitude ticks.

    Row 0 is the pole, so the label at colatitude t is 90 - t. The pole
    label is blanked by default to avoid crowding the centre.

    """
    labels = [str(int(90 - t)) for t in rad_ticks]
    if blank_pole and rad_ticks and rad_ticks[0] == 0:
        labels[0] = ''
    return labels

## Load data

Results pickles hold `{timestamp: {'ampere', 'predicted', 'input'}}` where
`predicted` is a raw `(2, 50, 24)` array — channel 0 mean, channel 1 standard
deviation. A missing file yields an empty dict rather than an error, so the
notebook still runs with a subset of the models available.

In [ ]:
def load_results(path, label):
    """Load a results pickle, returning {} and a note if it is absent."""
    if not os.path.exists(path):
        print(f'{label:12s} NOT FOUND at {path}')
        return {}
    with open(path, 'rb') as f:
        results = pickle.load(f)
    print(f'{label:12s} {len(results):,} timestamps')
    return results


acorn_results  = load_results(SCI_RESULTS, 'ACORN Sci')
op_results     = load_results(OP_RESULTS,  'ACORN Op')
bk_sci_results = load_results(BK_SCI_RESULTS, 'BK Sci')
bk_op_results  = load_results(BK_OP_RESULTS,  'BK Op')

# Solar wind record, used for the conditional analyses further down.
omni = pd.read_feather(
    os.path.join(SCI_CONFIG['data_dir'], 'sw_data', 'omni',
                 'omni_10_min_interp.feather')
)
print(f'\nOMNI: {len(omni):,} rows')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Gap-filling inference
# ═══════════════════════════════════════════════════════════════════════════
# Results files cover the test split only. These helpers run inference for
# timestamps that are missing -- typically the May 2023 storm window, which
# is held out as a case study -- and merge them in.
#
# Left as a function definition cell: nothing runs until the driver cell
# below is uncommented, since inference over a wide window is slow and
# fetches OMNI from NASA SPDF.


def load_ampere_lookup(config):
    """
    Load the cached AMPERE targets as {timestamp_str: grid}.

    Single cache now, shared by both models -- the targets do not depend
    on which model consumes them.
    """
    path = utils.ampere_file(config)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'AMPERE cache not found at {path}. Run data preparation first.'
        )
    with open(path, 'rb') as f:
        lookup = pickle.load(f)
    print(f'  {len(lookup):,} AMPERE timestamps loaded from {path}')
    return lookup


def fill_prediction_gaps(results, ampere_lookup, stime, etime, model_variant,
                         model_path=None, scaler_path=None):
    """
    Run ACORN inference over [stime, etime] and merge in any timestamps
    missing from `results` that also have an AMPERE target.

    Parameters
    ----------
    results       : dict  — existing results, modified and returned
    ampere_lookup : dict  — str(timestamp) -> AMPERE grid
    stime, etime  : timestamps bounding the window to fill
    model_variant : 'sci' or 'op'
    model_path, scaler_path : optional explicit artifact paths, for
        reproducing a specific historical run

    Note predict() returns FOUR values; the fourth is the scaled input
    sequence the model ran on, stored here as 'input' so downstream
    analysis can see exactly what produced each prediction.
    """
    starting_length = len(results)
    existing = set(results)

    model = FACInference(model_variant=model_variant,
                         model_path=model_path)
    mean, std, times, sequences = model.predict(start=str(stime), end=str(etime))

    added = 0
    for mu, sd, t, arr in zip(mean, std, times, sequences):
        key = str(pd.Timestamp(t))
        if key in existing or key not in ampere_lookup:
            continue
        results[key] = {
            'ampere':    ampere_lookup[key],
            'predicted': np.stack((mu, sd), axis=0),
            'input':     arr,
        }
        added += 1

    print(f'  {added:,} timestamps added ({starting_length:,} -> {len(results):,})')
    return results


def load_or_run_bk_inference(model_variant, model_path, ampere_lookup,
                             test_keys, save_path):
    """
    Load Kunduri (BK) baseline results, filling any gaps by inference.

    Loads from save_path if present, checks which of `test_keys` are
    missing, runs inference for those, then saves. Runs from scratch if
    the file does not exist.

    Parameters
    ----------
    model_variant : 'sci' or 'op'
    model_path    : path to the Keras .hdf5 baseline model
    ampere_lookup : dict — str(timestamp) -> AMPERE grid
    test_keys     : full set of required timestamps
    save_path     : load from / save to
    """
    results = {}
    if os.path.exists(save_path):
        with open(save_path, 'rb') as f:
            results = pickle.load(f)
        print(f'  {len(results):,} existing timestamps loaded')

        existing = {str(pd.Timestamp(k)) for k in results}
        missing = [k for k in test_keys if str(pd.Timestamp(k)) not in existing]
        if not missing:
            print(f'  No gaps — {len(results):,} timestamps ready.')
            return results
        print(f'  {len(missing)} missing — filling gaps...')
        keys_to_run = missing
    else:
        print(f'  No file at {save_path} — running full inference...')
        keys_to_run = test_keys

    times = pd.to_datetime(keys_to_run)
    bk_model = TFACInference(model_type=model_variant, model_path=model_path)

    # TFACInference returns (mean, None, times): the Kunduri model is
    # deterministic and has no uncertainty channel.
    mean_arr, _, valid_times = bk_model.predict(
        start=times.min().strftime('%Y-%m-%d %H:%M:%S'),
        end=times.max().strftime('%Y-%m-%d %H:%M:%S'),
    )

    added = 0
    for mu, t in zip(mean_arr, valid_times):
        key = str(pd.Timestamp(t))
        if key not in ampere_lookup:
            continue
        results[key] = {
            'ampere':    ampere_lookup[key],
            'predicted': mu,
            'std':       np.zeros_like(mu),   # placeholder: BK is deterministic
            'input':     None,
        }
        added += 1

    print(f'  {added:,} added — {len(results):,} total')

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, 'wb') as f:
        pickle.dump(results, f)
    print(f'  Saved to {save_path}')
    return results

In [ ]:
# ── Driver: uncomment to fill gaps and regenerate baseline results ────────
# Slow (fetches OMNI from NASA SPDF) and only needed when the results
# files lack the May 2023 window or the BK comparison.

ampere_lookup = load_ampere_lookup(SCI_CONFIG)
test_keys     = list(acorn_results)

acorn_results = fill_prediction_gaps(
    acorn_results, ampere_lookup, may_start, may_end, model_variant='sci')
op_results = fill_prediction_gaps(
    op_results, ampere_lookup, may_start, may_end, model_variant='op')

with open(SCI_RESULTS, 'wb') as f:
    pickle.dump(acorn_results, f)
with open(OP_RESULTS, 'wb') as f:
    pickle.dump(op_results, f)

bk_sci_results = load_or_run_bk_inference(
    'sci', 'models/FAC_BK_sci.hdf5', ampere_lookup, test_keys, BK_SCI_RESULTS)
bk_op_results = load_or_run_bk_inference(
    'op', 'models/FAC_BK_op.hdf5', ampere_lookup, test_keys, BK_OP_RESULTS)

del ampere_lookup
print(f'Sci {len(acorn_results):,}  Op {len(op_results):,}  '
      f'BK Sci {len(bk_sci_results):,}  BK Op {len(bk_op_results):,}')

## Load Weimer data

Weimer is an empirical model served by CCMC, on its own native grid, and is only
available for the May 2023 storm for this analysis.

It is handled at **two resolutions deliberately**: integrated currents are computed
at Weimer's native resolution using its own coordinates, while per-cell comparisons
use a bin-mean downsample onto the ACORN 50×24 grid.


In [ ]:
# Load Weimer (CCMC) results.
#
# Strategy:
#   - Total integrated current: computed at full Weimer resolution using its
#     own coordinate grids. No interpolation needed here.
#   - Per-cell comparisons (scatter, correlation): Weimer is downsampled to
#     the ACORN 50x24 grid by bin-mean averaging, so all comparisons are made
#     at AMPERE/ACORN resolution with no upsampling artefacts.

ACORN_LATS = np.arange(50)   # row indices 0-49
ACORN_MLTS = np.arange(24)   # MLT integer hours 0-23

weimer_df = pd.read_csv(
    '../data/weimer_05052023_09052023/weimer_may_2023.csv',
    parse_dates=['datetime']
)


def compute_weimer_integrated(group):
    """
    Compute total integrated current from a single Weimer timestamp at full
    resolution, using the actual LAT/MLT values from the file.
    """
    lat_vals = group['LAT'].to_numpy()
    mlt_vals = group['MLT'].to_numpy()
    fac_vals = group['FAC'].to_numpy()

    # Apply threshold
    fac_vals = np.where(np.abs(fac_vals) >= 0.1, fac_vals, 0.0)

    # Pivot to 2D so grid_surface_integral gets proper lat x MLT grids
    pivot = (
        pd.DataFrame({'LAT': lat_vals, 'MLT': mlt_vals, 'FAC': fac_vals})
          .pivot(index='LAT', columns='MLT', values='FAC')
          .sort_index(ascending=False)
    )
    lats = pivot.index.to_numpy()
    mlts = pivot.columns.to_numpy()
    mlat_grid = np.tile(lats.reshape(-1, 1), (1, len(mlts)))
    mlt_grid  = np.tile(mlts, (len(lats), 1))

    return grid_surface_integral(
        grid_lats=mlat_grid,
        grid_azis=mlt_grid,
        grid_values=np.abs(pivot.values),
        sphere_radius=6371200,
        aziunit='hour'
    )


def weimer_pivot(group):
    """
    Pivot a single Weimer timestamp group into a full-resolution LAT x MLT
    DataFrame, sorted descending by latitude (pole first).
    Returns a DataFrame with real MLAT values as index and real MLT values
    as columns.
    """
    return (
        pd.DataFrame({
            'LAT': group['LAT'].to_numpy(),
            'MLT': group['MLT'].to_numpy(),
            'FAC': group['FAC'].to_numpy(),
        })
        .pivot(index='LAT', columns='MLT', values='FAC')
        .sort_index(ascending=False)
    )


def downsample_weimer_to_acorn(weimer_fac_df):
    """
    Downsample a full-resolution Weimer FAC DataFrame onto the ACORN 50x24
    grid by taking the mean of all Weimer points that fall within each ACORN
    bin.  Used for all per-cell comparisons (scatter, correlation, RMSE) so
    that Weimer and AMPERE are compared on the same grid.

    ACORN grid:
      MLAT: bin centres at 89, 88, ..., 40 (edges at 89.5, 88.5, ..., 39.5)
      MLT : bin centres at 0, 1, ..., 23  (edges at -0.5, 0.5, ..., 23.5)

    Parameters
    ----------
    weimer_fac_df : pd.DataFrame -- full-res Weimer FAC, MLAT as index
                    (descending), MLT as columns (ascending)

    Returns
    -------
    pd.DataFrame (50, 24) with ACORN_LATS as index and ACORN_MLTS as columns.
    Empty bins are filled with 0.
    """
    _n_rows     = 40 if LAT_MASK_50 else 50
    acorn_mlats = 90 - np.arange(1, _n_rows + 1).astype(float)  # 89...(90-_n_rows)
    acorn_mlts  = np.arange(24).astype(float)                    # 0, 1, ..., 23

    w_lats = weimer_fac_df.index.to_numpy()
    w_mlts = weimer_fac_df.columns.to_numpy()

    # Assign each Weimer lat to the nearest ACORN lat bin
    lat_bin = np.argmin(
        np.abs(w_lats[:, None] - acorn_mlats[None, :]), axis=1
    )  # shape (n_weimer_lats,)

    # Assign each Weimer MLT to the nearest ACORN MLT bin
    mlt_diff = np.abs(w_mlts[:, None] - acorn_mlts[None, :])
    mlt_diff = np.minimum(mlt_diff, 24.0 - mlt_diff)
    mlt_bin  = np.argmin(mlt_diff, axis=1)  # shape (n_weimer_mlts,)

    # Accumulate bin sums and counts
    out_sum   = np.zeros((_n_rows, 24))
    out_count = np.zeros((_n_rows, 24))
    vals      = weimer_fac_df.values  # (n_weimer_lats, n_weimer_mlts)

    for li, lb in enumerate(lat_bin):
        for mi, mb in enumerate(mlt_bin):
            v = vals[li, mi]
            if np.isfinite(v):
                out_sum[lb, mb]   += v
                out_count[lb, mb] += 1

    with np.errstate(invalid='ignore'):
        out = np.where(out_count > 0, out_sum / out_count, 0.0)

    return pd.DataFrame(out, index=ACORN_LATS[:_n_rows], columns=ACORN_MLTS)


weimer_results    = {}
weimer_integrated = {}

for ts, group in tqdm.tqdm(weimer_df.groupby('datetime'), desc='Loading Weimer'):
    key = str(ts)

    # Full-resolution Weimer FAC pivot (real LAT x MLT)
    fac = weimer_pivot(group)
    fac = -fac  # Weimer FAC is sign-inverted relative to AMPERE
    fac = fac.fillna(0.0)

    # Full-resolution integrated current (uses own coordinate grids)
    integrated = compute_weimer_integrated(group)
    weimer_integrated[key] = integrated

    # Downsample Weimer to ACORN 50x24 grid for all per-cell comparisons.
    # AMPERE is already at ACORN resolution so no upsampling is needed.
    fac_acorn = downsample_weimer_to_acorn(fac)

    # Store AMPERE at native ACORN resolution (from acorn_results) so
    # all per-cell metrics compare on the same grid.
    # Always store ampere as a DataFrame at ACORN resolution.
    # acorn_results[key]['ampere'] is a DataFrame after preprocessing,
    # but may be a raw ndarray if this key ran before that loop.
    if key in acorn_results:
        _amp = acorn_results[key]['ampere']
        if isinstance(_amp, pd.DataFrame):
            ampere_acorn = _amp
        elif _amp.ndim == 1:
            ampere_acorn = pd.DataFrame(_amp.reshape(24, 50).T, index=ACORN_LATS, columns=ACORN_MLTS)
        else:
            ampere_acorn = pd.DataFrame(_amp, index=ACORN_LATS, columns=ACORN_MLTS)
    else:
        ampere_acorn = pd.DataFrame(np.zeros((50, 24)), index=ACORN_LATS, columns=ACORN_MLTS)

    weimer_results[key] = {
        'predicted':       fac_acorn,   # downsampled to ACORN grid (used for corr/RMSE/scatter)
        'predicted_full':  fac,         # full native resolution (used for polar frame plots)
        'ampere':          ampere_acorn,
        'total_integrated_flux': {'predicted': integrated}
    }

print(f'Loaded {len(weimer_results)} Weimer timestamps')


In [ ]:
# ── Weimer / AMPERE alignment diagnostic ────────────────────────────────────
# Run this after the Weimer loading cell to verify the MLT alignment between
# the full-resolution Weimer grid and the downsampled ACORN-resolution grid.
# Compares the MLT location of peak |FAC| in both representations.

diag_key = next(
    k for k in weimer_results
    if k in acorn_results and np.any(
        acorn_results[k]['ampere'].values
        if hasattr(acorn_results[k]['ampere'], 'values')
        else acorn_results[k]['ampere']
    )
)
print(f'Diagnostic timestamp: {diag_key}')

wfac_full = weimer_results[diag_key]['predicted_full']   # full native resolution
wfac_down = weimer_results[diag_key]['predicted']        # downsampled to ACORN 50x24
wamp      = acorn_results[diag_key]['ampere']            # AMPERE at ACORN resolution

# Normalise to DataFrames if needed
if not hasattr(wamp, 'columns'):
    wamp = pd.DataFrame(wamp.reshape(50, 24) if wamp.ndim == 1 else wamp)

# MLT column ranges
print(f'Weimer full-res MLT range   : {wfac_full.columns.min():.3f} -> {wfac_full.columns.max():.3f}')
print(f'Weimer downsampled MLT range: {wfac_down.columns.min():.0f} -> {wfac_down.columns.max():.0f}')
print(f'AMPERE MLT range            : {wamp.columns.min()} -> {wamp.columns.max()}')

# MLT of peak |FAC| in full-res Weimer
w_peak_col = int(np.argmax(np.abs(wfac_full.values).max(axis=0)))
w_peak_row = int(np.argmax(np.abs(wfac_full.values).max(axis=1)))
print(f'\nWeimer full-res peak |FAC|   : MLT={wfac_full.columns[w_peak_col]:.2f}  '
      f'MLAT={wfac_full.index[w_peak_row]:.1f}  '
      f'value={np.abs(wfac_full.values).max():.3f}')

# MLT of peak |FAC| in downsampled Weimer
wd_peak_col = int(np.argmax(np.abs(wfac_down.values).max(axis=0)))
wd_peak_row = int(np.argmax(np.abs(wfac_down.values).max(axis=1)))
print(f'Weimer downsampled peak |FAC|: MLT={wfac_down.columns[wd_peak_col]:.0f}  '
      f'row={wd_peak_row}  '
      f'value={np.abs(wfac_down.values).max():.3f}')

# MLT of peak |FAC| in AMPERE
a_peak_col = int(np.argmax(np.abs(wamp.values).max(axis=0)))
a_peak_row = int(np.argmax(np.abs(wamp.values).max(axis=1)))
print(f'AMPERE peak |FAC|            : MLT={wamp.columns[a_peak_col]}  '
      f'row={a_peak_row}  '
      f'value={np.abs(wamp.values).max():.3f}')

# Offset between full-res Weimer peak and downsampled Weimer peak
mlt_offset = wfac_full.columns[w_peak_col] - wfac_down.columns[wd_peak_col]
print(f'\nMLT offset full-res vs downsampled: {mlt_offset:.2f} hours')
print('If this is non-zero, the bin-mean downsampling has shifted the peak.')
print('Check downsample_weimer_to_acorn MLT wrap-around logic.')


## Apply threshold

AMPERE's inversion leaves low-amplitude speckle across the grid. Values below
0.1 µA/m² are zeroed, matching the `NOISE_FLOOR` applied during training so
evaluation and training treat the target identically.

In [ ]:
# # Apply 0.1 µA/m² threshold to ACORN and BK raw arrays before DataFrame conversion
# dicts = [op_results, acorn_results]
# for results in dicts:
#     for key in tqdm.tqdm(results.keys()):
#         results[key]['predicted'] = np.where(
#             np.abs(results[key]['predicted']) >= 0.1,
#             results[key]['predicted'], 0
#         )
#         results[key]['ampere'] = np.where(
#             np.abs(results[key]['ampere']) >= 0.1,
#             results[key]['ampere'], 0
#         )


## Helper functions

Integrated current and grid-handling utilities used throughout the rest of the
notebook.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Integrated current
# ═══════════════════════════════════════════════════════════════════════════
# The headline scalar metric: total current at
# a given time, in amperes. Obtained by area-weighting each grid cell and
# summing |FAC| over the whole grid.
#
# Area weighting matters. Cells near the pole represent far less area than
# cells at the equatorward edge, so an unweighted sum would badly overstate
# the polar contribution. grid_surface_integral handles this given real
# MLAT/MLT coordinates -- hence the care below about not reading coordinates
# off the DataFrame index.
def calculate_integrated_currents(
    data,
    radius=6371200,
    uncertainty=None,
    pos_or_neg='neither'
):
    """
    Compute the total integrated current from a (50x24) DataFrame.
    If uncertainty (n_samples x 50 x 24 array) is provided, returns (mean, std)
    over samples. Otherwise returns a scalar.

    Latitude grid is fixed to MLAT 89->40 degrees (matching the ACORN 50x24
    output grid), not derived from the DataFrame index, which stores row indices
    rather than real magnetic latitudes.
    """
    # Use fixed physical coordinate grids matching the (50, 24) ACORN output:
    #   MLT  : integer hours 0-23
    #   MLAT : magnetic latitude 89->40 degrees (row 0 = 89, row 49 = 40)
    # These must not be derived from the DataFrame index/columns, which store
    # row indices (0-49) rather than real latitudes, causing wrong area weights.
    # Derive grid dimensions from data shape so the function works for
    # both the full 50-row grid and the LAT_MASK_50 40-row grid.
    _n_lats    = data.shape[0]
    _mlt_vals  = np.arange(24)                          # MLT hours 0-23
    _lat_vals  = 90 - np.arange(1, _n_lats + 1)        # MLAT: 89...(90-n_lats)
    mlt_grid   = np.tile(_mlt_vals, (_n_lats, 1))                         # (n_lats, 24)
    mlat_grid  = np.tile(_lat_vals.reshape(-1, 1), (1, 24))               # (n_lats, 24)

    if not isinstance(uncertainty, np.ndarray):
        data = data.copy()
        data[np.abs(data) <= 0.1] = 0
        if pos_or_neg == 'pos':
            data[data < 0] = 0
        elif pos_or_neg == 'neg':
            data[data > 0] = 0
        return grid_surface_integral(
            grid_lats=mlat_grid, grid_azis=mlt_grid,
            grid_values=np.abs(data.values),
            sphere_radius=radius, aziunit='hour'
        )

    elif uncertainty.ndim == 3:
        int_flux = np.empty(uncertainty.shape[0])
        for n in range(uncertainty.shape[0]):
            sample = uncertainty[n].copy()
            sample[np.abs(sample) <= 0.1] = 0
            int_flux[n] = grid_surface_integral(
                grid_lats=mlat_grid, grid_azis=mlt_grid,
                grid_values=np.abs(sample),
                sphere_radius=radius, aziunit='hour'
            )
        return np.mean(int_flux), np.std(int_flux)

    else:
        raise ValueError('uncertainty must be a 3D array (n_samples x n_lats x 24)')


def splitting_omni_into_conditions(variable1, var_split, zero_split=False):
    '''
    divides the omni data by variable conditions

    Args:
        variable1 (str): variable to use to split the data
        var_split (foat, list[float]): value or list of values on which to split the data
        zero_split (bool, optional): whether to split the data on 0 in addition to the input values. Defaults to False.

    Returns:
        list[pd.DataFrames]: list of split dataframes. Ordered from lowest value bin to highest value bin.
    '''
    omni_local = pd.read_feather('~/data/sw_data/omni/omni_10_min_interp.feather')
    omni_local.dropna(inplace=True)
    split_df_list, split_names = [], []
    split_df_list.append(omni_local[omni_local[variable1] <= var_split[0]])
    split_names.append(f'{variable1} <= {var_split[0]}')
    if zero_split:
        split_df_list.append(omni_local[(omni_local[variable1] > var_split[0]) & (omni_local[variable1] <= 0)])
        split_df_list.append(omni_local[(omni_local[variable1] <= var_split[1]) & (omni_local[variable1] > 0)])
        split_names.append(f'{var_split[0]} < {variable1} <= 0')
        split_names.append(f'0 < {variable1} <= {var_split[1]}')
    else:
        split_df_list.append(omni_local[(omni_local[variable1] > var_split[0]) & (omni_local[variable1] <= var_split[1])])
        split_names.append(f'{var_split[0]} < {variable1} <= {var_split[1]}')
    split_df_list.append(omni_local[omni_local[variable1] > var_split[1]])
    split_names.append(f'{variable1} > {var_split[1]}')
    return split_df_list, split_names


def sample_from_gaussian(data, n_samples):
    """Sample from Gaussian distributions defined by a mean/std array."""
    if data.ndim == 3:
        means = np.abs(data[0])
        stds  = np.abs(data[1])
    elif data.ndim == 2:
        means = np.abs(data[:, 0].reshape(50, 24))
        stds  = np.abs(data[:, 1].reshape(50, 24))
    else:
        raise ValueError('data must be 2D or 3D')
    return np.random.normal(
        loc=means[np.newaxis],
        scale=stds[np.newaxis],
        size=(n_samples,) + means.shape
    )


## Data preprocessing

Splits the stored `(2, 50, 24)` predictions into labelled mean and standard
deviation DataFrames, normalises AMPERE onto the same grid, and attaches the
integrated-current scalars.

**Modifies the results dicts in place** — reload them before re-running.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Preprocessing: unpack predictions and derive per-timestamp quantities
# ═══════════════════════════════════════════════════════════════════════════
# Results files store 'predicted' as a raw (2, 50, 24) array: channel 0 is
# the mean, channel 1 the standard deviation. This loop splits those into
# labelled DataFrames, normalises the AMPERE target to the same shape, and
# attaches the integrated-current scalars used throughout the analysis.
#
# The May storm window gets extra treatment: 'spread' holds 100 Gaussian
# samples drawn per cell from (mean, std), which propagates the model's
# uncertainty through the area integral rather than integrating the mean
# alone. That is expensive, so it is computed only for the case-study window.
#
# Runs in place -- re-running after the dict has already been converted will
# fail, since 'predicted' is no longer a raw array. Reload the results first.
may_keys  = [
    key for key in acorn_results.keys()
    if may_start <= pd.to_datetime(key) <= may_end
]
# ── Preprocessing helper ─────────────────────────────────────────────────
# Both acorn_results (original) and acorn_results_addendum entries are now
# merged into acorn_results.  This loop handles both raw np.ndarray ampere
# (from the addendum) and pre-converted DataFrame ampere (from the main
# file) identically via the isinstance guard.

def to_ampere_df(amp):
    """Convert any AMPERE representation to a (50, 24) DataFrame."""
    if amp is None:
        return pd.DataFrame(np.zeros((50, 24)), columns=ACORN_MLTS, index=ACORN_LATS)
    if isinstance(amp, pd.DataFrame):
        return amp.reindex(index=ACORN_LATS, columns=ACORN_MLTS).fillna(0.0)
    arr = np.asarray(amp, dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(24, 50).T
    return pd.DataFrame(arr, columns=ACORN_MLTS, index=ACORN_LATS)


for key in tqdm.tqdm(op_results.keys()):
    in_may = key in may_keys
    op_results[key]['std'] = pd.DataFrame(
        op_results[key]['predicted'][1],
        columns=ACORN_MLTS, index=ACORN_LATS
    )
    if in_may:
        op_results[key]['spread'] = sample_from_gaussian(op_results[key]['predicted'], n_samples=100)
    op_results[key]['predicted'] = pd.DataFrame(
        op_results[key]['predicted'][0],
        columns=ACORN_MLTS, index=ACORN_LATS
    )
    op_results[key]['ampere'] = to_ampere_df(op_results[key].get('ampere'))
    op_results[key]['total_integrated_flux'] = {
        'ampere':    calculate_integrated_currents(op_results[key]['ampere']),
        'predicted': calculate_integrated_currents(op_results[key]['predicted']),
    }
    if in_may:
        op_results[key]['total_integrated_flux']['std'] = calculate_integrated_currents(
            data=op_results[key]['ampere'],
            uncertainty=op_results[key]['spread']
        )

for key in tqdm.tqdm(acorn_results.keys()):
    in_may = key in may_keys
    acorn_results[key]['std'] = pd.DataFrame(
        acorn_results[key]['predicted'][1],
        columns=ACORN_MLTS, index=ACORN_LATS
    )
    if in_may:
        acorn_results[key]['spread'] = sample_from_gaussian(acorn_results[key]['predicted'], n_samples=100)
    acorn_results[key]['predicted'] = pd.DataFrame(
        acorn_results[key]['predicted'][0],
        columns=ACORN_MLTS, index=ACORN_LATS
    )
    acorn_results[key]['ampere'] = to_ampere_df(acorn_results[key].get('ampere'))
    tif = {
        'ampere':    calculate_integrated_currents(acorn_results[key]['ampere']),
        'predicted': calculate_integrated_currents(acorn_results[key]['predicted']),
    }
    if in_may:
        tif['std'] = calculate_integrated_currents(
            data=acorn_results[key]['ampere'],
            uncertainty=acorn_results[key]['spread']
        )
        tif['pos_ampere']    = calculate_integrated_currents(acorn_results[key]['ampere'].copy(),    pos_or_neg='pos')
        tif['neg_ampere']    = calculate_integrated_currents(acorn_results[key]['ampere'].copy(),    pos_or_neg='neg')
        tif['pos_predicted'] = calculate_integrated_currents(acorn_results[key]['predicted'].copy(), pos_or_neg='pos')
        tif['neg_predicted'] = calculate_integrated_currents(acorn_results[key]['predicted'].copy(), pos_or_neg='neg')
    acorn_results[key]['total_integrated_flux'] = tif

# ── Apply latitude mask (optional) ──────────────────────────────────────
if LAT_MASK_50:
    _keep_rows = 40   # rows 0-39 → MLAT 89°..50°
    for _results in [acorn_results, op_results]:
        for _v in _results.values():
            for _field in ('predicted', 'std', 'ampere'):
                if _field in _v and hasattr(_v[_field], 'iloc'):
                    _v[_field] = _v[_field].iloc[:_keep_rows, :]
    print(f'LAT_MASK_50 applied: grids trimmed to {_keep_rows} rows (MLAT 89°->50°)')
else:
    print('LAT_MASK_50 off: using full 50-row grid (MLAT 89°->40°)')

# ── BK preprocessing ─────────────────────────────────────────────────────
# BK has no uncertainty (std=zeros) and no spread; predicted stored as raw
# (50,24) array rather than stacked (2,50,24).
for bk_results in [bk_sci_results, bk_op_results]:
    for key in tqdm.tqdm(bk_results.keys()):
        in_may = key in may_keys
        bk_results[key]['std'] = pd.DataFrame(
            bk_results[key]['std'] if isinstance(bk_results[key]['std'], np.ndarray)
            else np.zeros((50, 24)),
            columns=ACORN_MLTS, index=ACORN_LATS
        )
        _pred = bk_results[key]['predicted']
        if isinstance(_pred, np.ndarray) and _pred.ndim == 2:
            bk_results[key]['predicted'] = pd.DataFrame(
                _pred, columns=ACORN_MLTS, index=ACORN_LATS)
        bk_results[key]['ampere'] = to_ampere_df(bk_results[key].get('ampere'))
        bk_results[key]['total_integrated_flux'] = {
            'ampere':    calculate_integrated_currents(bk_results[key]['ampere']),
            'predicted': calculate_integrated_currents(bk_results[key]['predicted']),
        }

# Apply latitude mask to BK results
if LAT_MASK_50:
    _keep_rows = 40
    for _results in [bk_sci_results, bk_op_results]:
        for _v in _results.values():
            for _field in ('predicted', 'std', 'ampere'):
                if _field in _v and hasattr(_v[_field], 'iloc'):
                    _v[_field] = _v[_field].iloc[:_keep_rows, :]


## Create analysis DataFrames

Tidy per-model DataFrames for the May 2023 storm window, joined on timestamp so
models with partial coverage appear as NaN rather than truncating the index.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# May 2023 storm window: integrated current time series
# ═══════════════════════════════════════════════════════════════════════════
# Assembles one tidy DataFrame with a column per model, indexed by timestamp,
# for the storm case study. Models are joined rather than concatenated
# because coverage differs -- Weimer exists only for this window, so absent models
# appear as NaN instead of silently truncating the index.
# ── May storm window plotting DataFrame ──────────────────────────────────

op_results_keys = [key for key in op_results.keys()]

may_plotting_df = pd.DataFrame({
    'measured':      [acorn_results[key]['total_integrated_flux']['ampere']        for key in may_keys],
    'opp_mean':      [op_results[key]['total_integrated_flux']['predicted']        for key in may_keys],
    'opp_std':       [op_results[key]['total_integrated_flux']['std'][1]           for key in may_keys],
    'mean':          [acorn_results[key]['total_integrated_flux']['predicted']     for key in may_keys],
    'pos_ampere':    [acorn_results[key]['total_integrated_flux']['pos_ampere']    for key in may_keys],
    'neg_ampere':    [acorn_results[key]['total_integrated_flux']['neg_ampere']    for key in may_keys],
    'pos_predicted': [acorn_results[key]['total_integrated_flux']['pos_predicted'] for key in may_keys],
    'neg_predicted': [acorn_results[key]['total_integrated_flux']['neg_predicted'] for key in may_keys],
    'std':           [acorn_results[key]['total_integrated_flux']['std'][1]        for key in may_keys],
    'weimer':        [weimer_integrated.get(key, np.nan)                           for key in may_keys],
    'bk_sci':        [bk_sci_results[key]['total_integrated_flux']['predicted']   for key in may_keys],
    'bk_opp':        [bk_op_results[key]['total_integrated_flux']['predicted']   for key in may_keys],
}, index=[pd.to_datetime(key) for key in may_keys]).sort_index()

# Keys present in both ACORN results and Weimer results (the Weimer window)
weimer_keys = [k for k in acorn_results if k in weimer_results]

# ── All-timestamps integrated current DataFrames ──────────────────────────
opp_df = pd.DataFrame({
    'measured': [op_results[key]['total_integrated_flux']['ampere']    for key in op_results],
    'op':       [op_results[key]['total_integrated_flux']['predicted'] for key in op_results],
}, index=[pd.to_datetime(k) for k in op_results])

bk_sci_df = pd.DataFrame({
    'measured': [bk_sci_results[k]['total_integrated_flux']['ampere']    for k in bk_sci_results],
    'bk_sci':   [bk_sci_results[k]['total_integrated_flux']['predicted'] for k in bk_sci_results],
}, index=[pd.to_datetime(k) for k in bk_sci_results])

bk_opp_df = pd.DataFrame({
    'measured': [bk_op_results[k]['total_integrated_flux']['ampere']    for k in bk_op_results],
    'bk_opp':   [bk_op_results[k]['total_integrated_flux']['predicted'] for k in bk_op_results],
}, index=[pd.to_datetime(k) for k in bk_op_results])

sci_df = pd.DataFrame({
    'measured': [acorn_results[key]['total_integrated_flux']['ampere']             for key in acorn_results],
    'acorn':    [acorn_results[key]['total_integrated_flux']['predicted']          for key in acorn_results],
    'weimer':   [weimer_results[key]['total_integrated_flux']['predicted']
                 if key in weimer_results else np.nan                              for key in acorn_results],
}, index=[pd.to_datetime(k) for k in acorn_results])

# ── Weimer-window-only integrated current DataFrames ─────────────────────
opp_df_w = pd.DataFrame({
    'measured': [op_results[k]['total_integrated_flux']['ampere']    for k in weimer_keys],
    'op':       [op_results[k]['total_integrated_flux']['predicted'] for k in weimer_keys],
}, index=[pd.to_datetime(k) for k in weimer_keys])

sci_df_w = pd.DataFrame({
    'measured': [acorn_results[k]['total_integrated_flux']['ampere']    for k in weimer_keys],
    'acorn':    [acorn_results[k]['total_integrated_flux']['predicted'] for k in weimer_keys],
}, index=[pd.to_datetime(k) for k in weimer_keys])

# ── Flat per-cell DataFrames (all timestamps) ────────────────────────────
total_opp_df = pd.DataFrame({
    'measured': np.hstack([op_results[key]['ampere'].to_numpy().flatten()    for key in op_results]),
    'op':       np.hstack([op_results[key]['predicted'].to_numpy().flatten() for key in op_results]),
})

total_bk_sci_df = pd.DataFrame({
    'measured': np.hstack([bk_sci_results[k]['ampere'].to_numpy().flatten()    for k in bk_sci_results]),
    'bk_sci':   np.hstack([bk_sci_results[k]['predicted'].to_numpy().flatten() for k in bk_sci_results]),
})

total_bk_opp_df = pd.DataFrame({
    'measured': np.hstack([bk_op_results[k]['ampere'].to_numpy().flatten()    for k in bk_op_results]),
    'bk_opp':   np.hstack([bk_op_results[k]['predicted'].to_numpy().flatten() for k in bk_op_results]),
})

total_sci_df = pd.DataFrame({
    'measured': np.hstack([acorn_results[key]['ampere'].to_numpy().flatten()    for key in acorn_results]),
    'acorn':    np.hstack([acorn_results[key]['predicted'].to_numpy().flatten() for key in acorn_results]),
})

# ── Flat per-cell DataFrames (Weimer window only) ────────────────────────
total_opp_df_w = pd.DataFrame({
    'measured': np.hstack([op_results[k]['ampere'].to_numpy().flatten()    for k in weimer_keys]),
    'op':       np.hstack([op_results[k]['predicted'].to_numpy().flatten() for k in weimer_keys]),
})

total_sci_df_w = pd.DataFrame({
    'measured': np.hstack([acorn_results[k]['ampere'].to_numpy().flatten()    for k in weimer_keys]),
    'acorn':    np.hstack([acorn_results[k]['predicted'].to_numpy().flatten() for k in weimer_keys]),
})

# ── Weimer per-cell scatter (downsampled to ACORN grid) ──────────────────
weimer_scatter_df = pd.DataFrame({
    'measured': np.hstack([acorn_results[k]['ampere'].to_numpy().flatten()    for k in weimer_keys]),
    'weimer':   np.hstack([weimer_results[k]['predicted'].to_numpy().flatten() for k in weimer_keys]),
})


## Polar segment plotting

Animation frames across the storm. Colour scales are fixed across all frames
before plotting so intensity changes are real rather than an artefact of
per-frame autoscaling.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Animation frames: polar maps across the storm
# ═══════════════════════════════════════════════════════════════════════════
# Renders one PNG per timestep, later stitched into an animation.
#
# Colour scales are computed ONCE across every frame before any plotting, so
# the scale is fixed for the whole sequence. Per-frame autoscaling would make
# the storm appear to have constant intensity, since each frame would be
# normalised to its own maximum.
# ── Weimer lookup ─────────────────────────────────────────────────────────
weimer_fac_lookup = {
    pd.Timestamp(k): v['predicted_full']
    for k, v in weimer_results.items()
}

# ── Pre-compute global colour scales across ALL frames ────────────────────
print('Computing global colour scales...')
global_bwr_vals = []
global_std_vals = []

for key in tqdm.tqdm(may_keys, desc='scanning frames'):
    if key in acorn_results:
        global_bwr_vals.append(np.abs(acorn_results[key]['ampere'].values).max())
        global_bwr_vals.append(np.abs(acorn_results[key]['predicted'].values).max())
        global_std_vals.append(np.abs(acorn_results[key]['std'].values).max())
    if key in bk_sci_results:
        global_bwr_vals.append(np.abs(bk_sci_results[key]['predicted'].values).max())
    _wfac = weimer_fac_lookup.get(pd.Timestamp(key), None)
    if _wfac is not None:
        global_bwr_vals.append(np.nanmax(np.abs(_wfac.values)))

GLOBAL_BWR_VMAX = max(global_bwr_vals) if global_bwr_vals else 4.0
GLOBAL_STD_VMAX = max(global_std_vals) if global_std_vals else 1.5
print(f'  Global FAC vmax : {GLOBAL_BWR_VMAX:.4f}')
print(f'  Global std vmax : {GLOBAL_STD_VMAX:.4f}')


def plot_polar_segment(y_true, y_pred, std, bk_sci_pred, total_int_current_plot, title,
                       y_weimer=None,
                       bwr_vmax=3,
                       std_vmax=GLOBAL_STD_VMAX):
    """
    Plot a single timestep: integrated-current time series (top) and up to five
    polar FAC maps (bottom): Measured | ACORN Sci μ | ACORN Sci σ | Kunduri Sci | Weimer.
    All FAC panels share one symmetric bwr colorbar; σ has its own Purples colorbar.
    Colour scales are fixed globally across all frames for consistent video output.
    """
    theta_edges = np.linspace(0, 2*np.pi, 25)
    n_rad      = 40 if LAT_MASK_50 else 50
    lat_edges   = np.linspace(0, n_rad, n_rad + 1)
    th_mesh, r_mesh = np.meshgrid(theta_edges, lat_edges)

    rad_step   = max(1, n_rad // 5)
    rad_ticks   = list(range(0, n_rad, rad_step))
    rad_labels  = [''] + [str(int(80 - t)) for t in rad_ticks[:-1]]
    theta_ticks = np.linspace(0, 2*np.pi, 8, endpoint=False)

    has_weimer  = y_weimer is not None
    cols        = ['AMPERE', 'ACORN Sci', r'ACORN Sci $\sigma$', 'Kunduri Sci']
    col_cmaps   = ['bwr',    'bwr',       'Purples',              'bwr']
    if has_weimer:
        cols.append('Weimer')
        col_cmaps.append('bwr')
    n_cols_plot = len(cols)

    # ── Colour scales (globally fixed) ───────────────────────────────────
    bwr_norm  = mpl.colors.Normalize(vmin=-bwr_vmax, vmax=bwr_vmax)
    std_norm  = mpl.colors.Normalize(vmin=0, vmax=std_vmax)
    col_norms = [bwr_norm, bwr_norm, std_norm, bwr_norm]
    if has_weimer:
        col_norms.append(bwr_norm)

    # ── Layout ────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(4 * n_cols_plot, 10))
    gs  = fig.add_gridspec(2, 1, height_ratios=[1, 1], hspace=0.35)
    omni_axs  = fig.add_subplot(gs[0])
    polar_gs  = gs[1].subgridspec(1, n_cols_plot, wspace=0.1)
    polar_axs = [fig.add_subplot(polar_gs[0, i], projection='polar')
                 for i in range(n_cols_plot)]

    # ── Time-series panel ─────────────────────────────────────────────────
    omni_axs.set_title('Total Integrated Currents (May 2023)', fontsize=16)
    omni_axs.plot((total_int_current_plot['measured'].rolling(3).mean())/1e12,  label='Measured', color='black')
    if 'weimer' in total_int_current_plot.columns:
        omni_axs.plot((total_int_current_plot['weimer'].rolling(3).mean())/1e12, label='Weimer',           alpha=0.6, color='orange')
    omni_axs.plot((total_int_current_plot['mean'].rolling(3).mean())/1e12,       label=r'ACORN sci $\mu$', alpha=0.6, color='green')
    omni_axs.plot((total_int_current_plot['bk_sci'].rolling(3).mean())/1e12,     label=r'BK sci $\mu$',   alpha=0.6)
    omni_axs.axvline(pd.to_datetime(title), linestyle='-', color='black', linewidth=2)
    omni_axs.set_ylim(0)
    omni_axs.set_ylabel(r'Total Integrated Current (MA)', fontsize=13)
    omni_axs.margins(x=0)
    omni_axs.legend(fontsize=10)

    # ── Polar panels ──────────────────────────────────────────────────────
    def get_arr(arr):
        vals = arr.values if hasattr(arr, 'values') else np.asarray(arr)
        if vals.ndim == 1:
            vals = vals.reshape(24, 50).T
        return vals[:n_rad, :]

    if has_weimer:
        w_lats  = y_weimer.index.to_numpy()
        w_mlts  = y_weimer.columns.to_numpy()
        w_radii = (90 - w_lats)
        wr      = np.tile(w_radii.reshape(-1, 1), (1, len(w_mlts)))
        wth     = np.tile((w_mlts / 24) * 2 * np.pi, (len(w_lats), 1))

    panel_data = [
        ('AMPERE',              get_arr(y_true)),
        ('ACORN Sci',           get_arr(y_pred)),
        (r'ACORN Sci $\sigma$', get_arr(std)),
        ('Kunduri Sci',              get_arr(bk_sci_pred)),
    ]
    if has_weimer:
        panel_data.append(('Weimer', None))

    for i, (ax, (col_label, cmap, norm), (_, arr)) in enumerate(
        zip(polar_axs, zip(cols, col_cmaps, col_norms), panel_data)
    ):
        if col_label == 'Weimer':
            ax.pcolormesh(wth, wr, y_weimer.values, cmap=cmap, norm=norm)
        else:
            ax.pcolormesh(th_mesh, r_mesh, arr, cmap=cmap, norm=norm)

        polar_setup(ax, n_rad, rad_ticks=rad_ticks, rad_labels=rad_labels,
                    mlt_fontsize=10, rad_fontsize=10, grid_alpha=0.4)
        # ax.set_ylim(0, n_rad)
        # ax.set_ylim(0, 15)
        ax.set_title(col_label, fontsize=15)

    # ── Colorbars ─────────────────────────────────────────────────────────
    mu_cbar = fig.colorbar(
        mpl.cm.ScalarMappable(norm=bwr_norm, cmap='bwr'),
        ax=polar_axs[0], orientation='vertical', fraction=0.05, pad=0.02, location='left'
    )
    mu_cbar.set_label(r'FAC ($\mu$A/m²)', size=14)
    mu_cbar.ax.tick_params(labelsize=12)

    std_cbar = fig.colorbar(
        mpl.cm.ScalarMappable(norm=std_norm, cmap='Purples'),
        ax=polar_axs[4], orientation='vertical', fraction=0.05, pad=0.02
    )
    std_cbar.set_label(r'Uncertainty ($\mu$A/m²)', size=14)
    std_cbar.ax.tick_params(labelsize=12)

    ts_label = pd.Timestamp(title).strftime('%Y-%m-%d %H:%M')
    fig.suptitle(f'FAC Polar Maps — {ts_label}', fontsize=18, fontweight='bold', y=0.98)

    os.makedirs(f'plots/may_2023_storm_zoomed_{RUN_TAG}', exist_ok=True)
    plt.savefig(
        f'plots/may_2023_storm_zoomed_{RUN_TAG}/{title.replace(":", ";").replace(" ", "_")}.png',
        dpi=150
    )
    plt.close(fig)

# # ── Render frames ─────────────────────────────────────────────────────────
# for key in tqdm.tqdm(may_keys):
#     # _fn = f'plots/may_2023_storm_{RUN_TAG}/{key.replace(":", ";").replace(" ", "_")}.png'
#     # if os.path.exists(_fn):
#     #     continue

#     weimer_fac  = weimer_fac_lookup.get(pd.Timestamp(key), None)
#     _n_out      = 40 if LAT_MASK_50 else 50
#     bk_sci_pred = (bk_sci_results[key]['predicted'].values
#                    if key in bk_sci_results
#                    else np.zeros((_n_out, 24)))

#     plot_polar_segment(
#         acorn_results[key]['ampere'].values,
#         acorn_results[key]['predicted'].values,
#         acorn_results[key]['std'].values,
#         bk_sci_pred,
#         may_plotting_df,
#         title=key,
#         y_weimer=weimer_fac,
#     )

In [ ]:
# import cv2

# def create_video_from_pngs_opencv(image_folder, output_video_path, fps,
#                                    image_prefix='2023', image_extension='.png'):
#     images = sorted([
#         img for img in os.listdir(image_folder)
#         if img.startswith(image_prefix) and img.endswith(image_extension)
#     ])

#     if not images:
#         print(f'No images found in {image_folder} with prefix '
#               f'"{image_prefix}" and extension "{image_extension}".')
#         return

#     # Read first valid frame to get dimensions
#     first_frame = None
#     for img_name in images:
#         first_frame = cv2.imread(os.path.join(image_folder, img_name))
#         if first_frame is not None:
#             break
#     if first_frame is None:
#         print('Could not read any images — aborting.')
#         return

#     height, width = first_frame.shape[:2]
#     print(f'Frame size: {width}x{height}  |  {len(images)} frames  |  {fps} fps')
#     print(f'Estimated duration: {len(images)/fps:.1f}s')

#     # Try H.264 first, fall back to mp4v
#     fourcc = cv2.VideoWriter_fourcc(*'avc1')
#     out    = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
#     if not out.isOpened():
#         print('avc1 codec unavailable, falling back to mp4v')
#         fourcc = cv2.VideoWriter_fourcc(*'mp4v')
#         out    = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

#     skipped = 0
#     try:
#         for img_name in tqdm.tqdm(images, desc='Writing frames'):
#             frame = cv2.imread(os.path.join(image_folder, img_name))
#             if frame is None:
#                 print(f'  Warning: could not read {img_name} — skipping')
#                 skipped += 1
#                 continue
#             # Resize if this frame differs from the expected dimensions
#             if frame.shape[:2] != (height, width):
#                 frame = cv2.resize(frame, (width, height))
#             out.write(frame)
#     finally:
#         out.release()

#     print(f'Video saved to {output_video_path}')
#     if skipped:
#         print(f'  {skipped} frames skipped due to read errors')


# create_video_from_pngs_opencv(
#     image_folder      = f'plots/may_2023_storm_zoomed_{RUN_TAG}',
#     output_video_path = f'plots/may_2023_storm_zoomed_{RUN_TAG}.mp4',
#     fps               = 24,
#     image_prefix      = '2023',
# )

## Time series plots

Integrated current through the storm, model against observation.

In [ ]:
sample_ts = [
    '2023-05-06 04:00:00',
    '2023-05-06 12:00:00',
    '2023-05-07 08:00:00',
    '2023-05-08 01:30:00'
]

fig, ax = plt.subplots(1, 1, figsize=(25, 10))
ax.set_title('Total Integrated Currents - May 2023', fontsize=20)
ax.plot(may_plotting_df['measured'].rolling(5).mean(), label='Measured')
ax.plot(may_plotting_df['mean'].rolling(5).mean(),     label=r'ACORN Sci $\mu$', alpha=0.6, color='orange')
ax.plot(may_plotting_df['bk_sci'].rolling(5).mean(), label=r'Kunduri Sci', alpha=0.6, color='green')
ax.plot(may_plotting_df['weimer'].rolling(5).mean(),   label='Weimer',            alpha=0.6, color='purple')

# ── Vertical lines at polar plot timestamps ───────────────────────────────
# for ts in sample_ts:
#     ax.axvline(pd.Timestamp(ts), color='grey', linewidth=1, linestyle='--', alpha=0.7)
# ax.axvline(pd.Timestamp(sample_ts[0]), color='grey', linewidth=1,
#            linestyle='--', alpha=0.7, label='Polar snapshots')

ax.set_ylim(0, may_plotting_df['measured'].max() * 1.1)
ax.set_xlabel('Time')
ax.set_ylabel(r'Total Integrated Current ($\mu$A)', fontsize=15)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
plt.xticks(may_plotting_df.index[::350], rotation=30)
ax.margins(x=0)
plt.legend()
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Static polar comparison panels
# ═══════════════════════════════════════════════════════════════════════════
# One row per timestamp, five columns: AMPERE observation, ACORN Sci mean,
# ACORN Sci uncertainty, the Kunduri baseline, and Weimer.
#
# The sigma column uses a sequential colormap (Purples) because uncertainty
# is one-sided; the FAC columns use a diverging map (bwr) centred on zero so
# upward and downward currents read symmetrically.
def plot_polar_timesteps(
    timestamps, acorn_results, op_results, weimer_results, bk_sci_results,
    n_cols=4, figsize_per_panel=(4, 4), save_path=None
):
    """
    Polar FAC maps: one row per timestamp, 5 columns:
    AMPERE | ACORN Sci | ACORN Sci σ | BK Sci | Weimer
    """
    cols        = ['AMPERE', 'ACORN Sci', r'ACORN Sci $\sigma$', 'Kunduri Sci', 'Weimer']
    col_cmaps   = ['bwr',    'bwr',       'Purples',              'bwr',    'bwr']
    n_cols_plot = len(cols)

    pages = [timestamps[i:i+n_cols] for i in range(0, len(timestamps), n_cols)]

    theta_edges = np.linspace(0, 2*np.pi, 25)
    n_rad      = 40 if LAT_MASK_50 else 50
    lat_edges   = np.linspace(0, n_rad, n_rad + 1)
    th_mesh, r_mesh = np.meshgrid(theta_edges, lat_edges)

    rad_step   = max(1, n_rad // 5)
    rad_ticks   = list(range(0, n_rad, rad_step))
    rad_labels  = ['']+[str(int(80 - t)) for t in rad_ticks[:-1]]
    theta_ticks = np.linspace(0, 2*np.pi, 8, endpoint=False)

    figs = []

    for p_idx, page_ts in enumerate(pages):
        n_t = len(page_ts)
        fig, axs = plt.subplots(
            n_t, n_cols_plot,
            figsize=(figsize_per_panel[0] * n_cols_plot,
                    figsize_per_panel[1] * n_t),
            subplot_kw=dict(projection='polar'),
        )
        fig.subplots_adjust(hspace=-0.3, wspace=0.1)
        if n_t == 1:
            axs = axs[np.newaxis, :]

        bwr_vals, std_vals = [], []
        for ts in page_ts:
            if ts in acorn_results:
                bwr_vals.append(np.abs(acorn_results[ts]['ampere'].to_numpy()).max())
                bwr_vals.append(np.abs(acorn_results[ts]['predicted'].to_numpy()).max())
                std_vals.append(np.abs(acorn_results[ts]['std'].to_numpy()).max())
            if ts in bk_sci_results:
                bwr_vals.append(np.abs(bk_sci_results[ts]['predicted'].to_numpy()).max())
            if ts in weimer_results:
                bwr_vals.append(np.abs(weimer_results[ts]['predicted'].to_numpy()).max())

        bwr_vmax  = max(bwr_vals) if bwr_vals else 1.0
        std_vmax  = max(std_vals) if std_vals else 1.0
        bwr_norm  = mpl.colors.Normalize(vmin=-bwr_vmax, vmax=bwr_vmax)
        std_norm  = mpl.colors.Normalize(vmin=0, vmax=std_vmax)
        col_norms = [bwr_norm, bwr_norm, std_norm, bwr_norm, bwr_norm]

        for row, ts in enumerate(page_ts):
            ts_label = pd.Timestamp(ts).strftime('%m-%d %H:%M')

            def _get(results, field):
                if ts not in results:
                    return np.zeros((n_rad, 24))
                arr  = results[ts][field]
                vals = arr.values if hasattr(arr, 'values') else np.asarray(arr)
                if vals.ndim == 1:
                    vals = vals.reshape(24, 50).T
                return vals[:n_rad, :]

            data = {
                'AMPERE':              _get(acorn_results,  'ampere'),
                'ACORN Sci':           _get(acorn_results,  'predicted'),
                r'ACORN Sci $\sigma$': _get(acorn_results,  'std'),
                'Kunduri Sci':              _get(bk_sci_results, 'predicted'),
                'Weimer':              _get(weimer_results,  'predicted'),
            }

            for col, (col_label, cmap, norm) in enumerate(
                zip(cols, col_cmaps, col_norms)
            ):
                ax  = axs[row, col]
                arr = data[col_label]
                ax.pcolormesh(th_mesh, r_mesh, arr, cmap=cmap, norm=norm)
                polar_setup(ax, n_rad, rad_ticks=rad_ticks, rad_labels=rad_labels,
                            mlt_fontsize=10, rad_fontsize=10, grid_alpha=0.4)
                ax.set_ylim(0, n_rad)
                if row == 0:
                    ax.set_title(col_label, fontsize=18)
                if col == 0:
                    ax.set_ylabel(ts_label, fontsize=15, labelpad=15)

        mu_cbar = fig.colorbar(
            mpl.cm.ScalarMappable(norm=bwr_norm, cmap='bwr'),
            ax=axs.ravel().tolist(), orientation='vertical', fraction=0.04, pad=0.07
        )
        mu_cbar.set_label(r'FAC ($\mu$A/m²)', size=19)
        mu_cbar.ax.tick_params(labelsize=19)
        std_cbar = fig.colorbar(
            mpl.cm.ScalarMappable(norm=std_norm, cmap='Purples'),
            ax=axs.ravel().tolist(), orientation='vertical', fraction=0.045, pad=0.04
        )
        std_cbar.set_label(r'Uncertainty ($\mu$A/m²)', size=19)
        std_cbar.ax.tick_params(labelsize=19)

        n = fig.suptitle('FAC Polar Maps', fontsize=25, fontweight='bold', y=0.9)
        figs.append(fig)

        if save_path:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            if len(pages) == 1:
                plt.savefig(save_path, dpi=150, bbox_inches='tight')
            else:
                base, ext = os.path.splitext(save_path)
                plt.savefig(f'{base}_p{p_idx+1}{ext}', dpi=150, bbox_inches='tight')
        plt.show()

    return figs


# ── Total integrated current time series ──────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(15, 7))
ax.set_title('Total Integrated Currents May 2023', fontsize=30, fontweight='bold')
ax.plot(may_plotting_df['measured'].rolling(3).mean()/1e12, label='Measured',           color='black')
ax.plot(may_plotting_df['weimer'].rolling(3).mean()/1e12,   label='Weimer',            alpha=0.7, color='orange')
ax.plot(may_plotting_df['mean'].rolling(3).mean()/1e12,     label=r'ACORN sci $\mu$', alpha=0.7, color='green')
ax.plot(may_plotting_df['bk_sci'].rolling(3).mean()/1e12,     label=r'BK sci', alpha=0.9)

for ts in sample_ts:
    ax.axvline(pd.Timestamp(ts), color='grey', linewidth=1, linestyle='--', alpha=0.7)
ax.axvline(pd.Timestamp(sample_ts[0]), color='grey', linewidth=2,
           linestyle='--', alpha=0.7, label='Polar snapshots')

ax.set_ylim(0, may_plotting_df['measured'].max()/1e12 * 1.1)
ax.set_ylabel(r'Total Integrated Current (MA)', fontsize=20)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
plt.xticks(may_plotting_df.index[::350], rotation=30, fontsize=15)
plt.yticks(fontsize=15)
ax.margins(x=0)
plt.legend(fontsize=13)
plt.tight_layout()
plt.savefig('plots/total_int_currents_may_2023.png')
plt.show()

figs = plot_polar_timesteps(
    sample_ts, acorn_results, op_results, weimer_results, bk_sci_results,
    n_cols=4,
    save_path=f'plots/polar_timesteps_{RUN_TAG}.png'
)


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# ── Load and display two figures side by side ────────────────────────────
FIG_BOTTOM    = f'plots/polar_timesteps_{RUN_TAG}.png'
FIG_TOP   = 'plots/total_int_currents_may_2023.png'

img_bottom  = np.array(Image.open(FIG_BOTTOM))
img_top = np.array(Image.open(FIG_TOP))

fig, axs = plt.subplots(2,1, figsize=(15,25), height_ratios=[5,7])

axs[1].imshow(img_bottom)
# axs[1].set_title(TITLE_RIGHT, fontsize=14)
axs[1].axis('off')

axs[0].imshow(img_top)
# axs[0].set_title(TITLE_LEFT, fontsize=14)
axs[0].axis('off')

plt.tight_layout()

plt.subplots_adjust(wspace=-0.1, hspace=-0.25)
plt.savefig('plots/may_2023_predictions.png', bbox_inches='tight')

## Scatter plots

Predicted against observed integrated current as 2D histograms — a scatter
saturates at this sample count and hides the density structure.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Predicted vs observed integrated current -- full test set
# ═══════════════════════════════════════════════════════════════════════════
# 2D histograms rather than scatter: with this many points a scatter plot
# saturates and hides the density structure. Log colour normalisation keeps
# the sparse high-current tail visible against the dense quiet-time core.
#
# All four panels share x and y limits so the models can be compared by eye.
# ── Figure A: Full test set ──────────────────────────────────────────────
fig, axs = plt.subplots(ncols=4, nrows=1, figsize=(20, 7), constrained_layout=True, sharey=True)
fig.supylabel(r'Total Integrated Currents (MA)', fontsize=28, fontweight='bold', x=1, rotation=-90)

for ax, ycol, src_df, title, v_scale in [
    (axs[0], 'op',    opp_df,    'ACORN Op',    0.25e2),
    (axs[1], 'acorn',  sci_df,    'ACORN Sci',    0.25e2),
    (axs[2], 'bk_opp', bk_opp_df, 'Kunduri Op',  0.25e2),
    (axs[3], 'bk_sci', bk_sci_df, 'Kunduri Sci',  0.25e2),
]:
    df_plot = src_df[['measured', ycol]].dropna()
    ax.hist2d(df_plot['measured']/1e12, df_plot[ycol]/1e12, bins=50, norm=colors.LogNorm(), cmap='magma')
    ax.set_xlim(1.7, v_scale)
    ax.set_ylim(0, 0.22e2)
    ax.set_xlabel('Measured', fontsize=20)
    if ax==axs[0]:
        ax.set_ylabel('Predicted', fontsize=20)
    ax.set_title(title, fontsize=26)
    ax.tick_params(labelsize=15)
    lo = min(df_plot['measured'].min(), df_plot[ycol].min())
    hi = max(df_plot['measured'].max(), df_plot[ycol].max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=2)
    nrmse = np.round(np.sqrt(MSE(df_plot['measured'], df_plot[ycol])) / df_plot['measured'].std(), 3)
    corr  = round(df_plot['measured'].corr(df_plot[ycol]), 3)
    ax.text(0.02, 0.95, f'NRMSE: {nrmse}\nCorr: {corr}',
            transform=ax.transAxes, fontsize=20,
            horizontalalignment='left', verticalalignment='top',
            bbox=dict(facecolor='white', alpha=1))
plt.savefig(f'plots/model_results/scatter_integrated_all_{RUN_TAG}.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure B: Weimer window ───────────────────────────────────────────────
fig, axs = plt.subplots(ncols=5, nrows=1, figsize=(25, 7), constrained_layout=True, sharey=True)
fig.supylabel(r'Total Integrated Currents (MA)', fontsize=28, fontweight='bold', x=1, rotation=-90)

bk_sci_df_w = bk_sci_df[bk_sci_df.index.isin(pd.to_datetime(weimer_keys))]
bk_opp_df_w = bk_opp_df[bk_opp_df.index.isin(pd.to_datetime(weimer_keys))]

for ax, ycol, src_df, title, v_scale in [
    (axs[0], 'op',    opp_df_w,    'ACORN Op',    0.18e2),
    (axs[1], 'acorn',  sci_df_w,    'ACORN Sci',    0.18e2),
    (axs[2], 'bk_opp', bk_opp_df_w, 'Kunduri Op',  0.18e2),
    (axs[3], 'bk_sci', bk_sci_df_w, 'Kunduri Sci',  0.18e2),
    (axs[4], 'weimer', sci_df,       'Weimer',      0.18e2),
]:
    df_plot = src_df[['measured', ycol]].dropna()
    ax.hist2d(df_plot['measured']/1e12, df_plot[ycol]/1e12, bins=50, norm=colors.LogNorm(), cmap='magma')
    ax.set_xlim(1.7, v_scale)
    ax.set_ylim(0, 0.2e2)
    ax.set_xlabel('Measured', fontsize=20)
    if ax==axs[0]:
        ax.set_ylabel('Predicted', fontsize=20)
    ax.set_title(title, fontsize=26)
    ax.tick_params(labelsize=15)
    lo = min(df_plot['measured'].min(), df_plot[ycol].min())/1e12
    hi = max(df_plot['measured'].max(), df_plot[ycol].max())/1e12
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=2)
    nrmse = np.round(np.sqrt(MSE(df_plot['measured'], df_plot[ycol])) / df_plot['measured'].std(), 3)
    corr  = round(df_plot['measured'].corr(df_plot[ycol]), 3)
    ax.text(0.02, 0.95, f'NRMSE: {nrmse}\nCorr: {corr}',
            transform=ax.transAxes, fontsize=20,
            horizontalalignment='left', verticalalignment='top',
            bbox=dict(facecolor='white', alpha=1))
plt.savefig(f'plots/model_results/scatter_integrated_weimer_{RUN_TAG}.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Figure A: Full test set ──────────────────────────────────────────────
fig, axs = plt.subplots(ncols=4, nrows=1, figsize=(20, 7), constrained_layout=True, sharey=True)
fig.supylabel(r'FAC Predictions ($\mu$A/$m^2$)', fontsize=28, fontweight='bold', x=1, rotation=-90)

for ax, ycol, src_df, title, v_scale, n_bins in [
    (axs[0], 'op',    total_opp_df,    'ACORN Op',    5, (100,30)),
    (axs[1], 'acorn',  total_sci_df,    'ACORN Sci',    5, (100,30)),
    (axs[2], 'bk_opp', total_bk_opp_df, 'Kunduri Op',  5, (100,30)),
    (axs[3], 'bk_sci', total_bk_sci_df, 'Kunduri Sci',  5, (100,30)),
]:
    df_plot = src_df[['measured', ycol]].dropna()
    ax.hist2d(df_plot['measured'], df_plot[ycol], bins=n_bins, norm=colors.LogNorm(), cmap='magma')
    ax.set_xlim(-v_scale, v_scale)
    ax.set_xlabel('Measured', fontsize=20)
    if ax==axs[0]:
        ax.set_ylabel('Predicted', fontsize=20)
    ax.set_title(title, fontsize=26)
    ax.set_xlim(-5,5)
    ax.set_ylim(-1.9,1.9)
    ax.tick_params(labelsize=15)
    lo = min(df_plot['measured'].min(), df_plot[ycol].min())
    hi = max(df_plot['measured'].max(), df_plot[ycol].max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=2)
    nrmse = np.round(np.sqrt(MSE(df_plot['measured'], df_plot[ycol])) / df_plot['measured'].std(), 3)
    corr  = round(df_plot['measured'].corr(df_plot[ycol]), 3)
    ax.text(0.6, 0.11, f'NRMSE: {nrmse}\nCorr: {corr}',
            transform=ax.transAxes, fontsize=17,
            horizontalalignment='left', verticalalignment='top',
            bbox=dict(facecolor='white', alpha=1))
plt.savefig(f'plots/model_results/scatter_fac_all_{RUN_TAG}.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure B: Weimer window ───────────────────────────────────────────────
fig, axs = plt.subplots(ncols=5, nrows=1, figsize=(25, 7), constrained_layout=True, sharey=True)
fig.supylabel(r'FAC Predictions ($\mu$A/$m^2$)', fontsize=28, fontweight='bold', x=1, rotation=-90)

wk_set = set(weimer_keys)
total_bk_sci_df_w = pd.DataFrame({
    'measured': np.hstack([bk_sci_results[k]['ampere'].to_numpy().flatten()    for k in bk_sci_results if k in wk_set]),
    'bk_sci':   np.hstack([bk_sci_results[k]['predicted'].to_numpy().flatten() for k in bk_sci_results if k in wk_set]),
})
total_bk_opp_df_w = pd.DataFrame({
    'measured': np.hstack([bk_op_results[k]['ampere'].to_numpy().flatten()    for k in bk_op_results if k in wk_set]),
    'bk_opp':   np.hstack([bk_op_results[k]['predicted'].to_numpy().flatten() for k in bk_op_results if k in wk_set]),
})

for ax, ycol, src_df, title, v_scale, n_bins in [
    (axs[0], 'op',    total_opp_df_w,    'ACORN Op',      5, 30),
    (axs[1], 'acorn',  total_sci_df_w,    'ACORN Sci',      5, 30),
    (axs[2], 'bk_opp', total_bk_opp_df_w, 'Kunduri Op',    5, 30),
    (axs[3], 'bk_sci', total_bk_sci_df_w, 'Kunduri Sci',    5, 30),
    (axs[4], 'weimer', weimer_scatter_df,  'Weimer',        5, 30),
]:
    df_plot = src_df[['measured', ycol]].dropna()
    ax.hist2d(df_plot['measured'], df_plot[ycol], bins=n_bins, norm=colors.LogNorm(), cmap='magma')
    ax.set_xlim(-v_scale, v_scale)
    ax.set_xlabel('Measured', fontsize=20)
    if ax==axs[0]:
        ax.set_ylabel('Predicted', fontsize=20)
    ax.set_title(title, fontsize=26)
    ax.set_xlim(-5,5)
    ax.set_ylim(-1.7,1.7)
    ax.tick_params(labelsize=15)
    lo = min(df_plot['measured'].min(), df_plot[ycol].min())
    hi = max(df_plot['measured'].max(), df_plot[ycol].max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=2)
    nrmse = np.round(np.sqrt(MSE(df_plot['measured'], df_plot[ycol])) / df_plot['measured'].std(), 3)
    corr  = round(df_plot['measured'].corr(df_plot[ycol]), 3)
    ax.text(0.6, 0.11, f'NRMSE: {nrmse}\nCorr: {corr}',
            transform=ax.transAxes, fontsize=17,
            horizontalalignment='left', verticalalignment='top',
            bbox=dict(facecolor='white', alpha=1))
plt.savefig(f'plots/model_results/scatter_fac_weimer_{RUN_TAG}.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# ── Full test set ─────────────────────────────────────────────────────────
fig, axs = plt.subplots(2, 1, figsize=(11, 8))
fig.suptitle('Predicted vs. Measured — Full Test Set (271 Days)', fontsize=18, fontweight='bold')

for ax, path, title in [
    (axs[0], f'plots/model_results/scatter_integrated_all_{RUN_TAG}.png', 'Total Integrated Current (MA)'),
    (axs[1], f'plots/model_results/scatter_fac_all_{RUN_TAG}.png',        r'FAC Density ($\muA/m^2$)'),
]:
    ax.imshow(np.array(Image.open(path)))
    # ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.subplots_adjust(wspace=0.02)
plt.savefig(f'plots/model_results/predicted_vs_measured_all_{RUN_TAG}.png', bbox_inches='tight', dpi=150)
plt.show()

# ── Weimer window ─────────────────────────────────────────────────────────
fig, axs = plt.subplots(2, 1, figsize=(13, 8))
fig.suptitle('Predicted vs. Measured — Test Storm (May 2023)', fontsize=18, fontweight='bold')

for ax, path, title in [
    (axs[0], f'plots/model_results/scatter_integrated_weimer_{RUN_TAG}.png', 'Total Integrated Current (MA)'),
    (axs[1], f'plots/model_results/scatter_fac_weimer_{RUN_TAG}.png',        r'FAC Density ($\muA/m^2$)'),
]:
    ax.imshow(np.array(Image.open(path)))
    # ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.subplots_adjust(wspace=0.02)
plt.savefig(f'plots/model_results/predicted_vs_measured_may_2023_{RUN_TAG}.png', bbox_inches='tight', dpi=150)
plt.show()

## Correlation analysis

Per-bin Pearson correlation between prediction and observation across all
timestamps. Each bin answers "does the model track this region over time",
not "does one map match".

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Spatial correlation tables
# ═══════════════════════════════════════════════════════════════════════════
# Pearson correlation between prediction and observation, computed per
# spatial bin across all timestamps -- so each cell answers "how well does
# the model track this region over time", not "how well does one map match".
#
# Binning coarser than the native grid (default 10 deg MLAT x 3 h MLT) is
# deliberate: single-cell correlations are noisy given the sample count.
#
# Grid dimensions are read from the first entry rather than hardcoded, so the
# same function serves the 50x24 ACORN grid and Weimer's finer native grid.
def creating_correlation_tables(results_dict, lat_bin_width=10, mlt_bin_width=3, mlt_bin_start=23):
    """
    Compute per-bin Pearson correlation between 'ampere' and 'predicted' across
    all timestamps in results_dict.

    Works at whatever resolution the DataFrames carry (50x24 for ACORN/BK,
    full Weimer resolution for Weimer).  Grid dimensions are read from the
    first entry rather than being hardcoded.
    """
    keys = list(results_dict.keys())
    if not keys:
        return pd.DataFrame()
    n_lats  = results_dict[keys[0]]['predicted'].shape[0]
    n_mlts  = results_dict[keys[0]]['predicted'].shape[1]
    # mlt_bin_start=23 was designed for the 24-column ACORN grid (wraps at 24).
    # For Weimer's finer MLT grid we start at 0 to cover all columns.
    effective_start = 0 if n_mlts > 24 else mlt_bin_start
    max_mlt = effective_start + n_mlts

    def _to_arr(x):
        return x.values if hasattr(x, 'values') else np.asarray(x)
    ampere_stack     = np.stack([_to_arr(results_dict[k]['ampere'])    for k in keys], axis=0)
    prediction_stack = np.stack([_to_arr(results_dict[k]['predicted']) for k in keys], axis=0)

    mlt_ranges = {
        mlt_bin: [(i % n_mlts) for i in range(mlt_bin, mlt_bin + mlt_bin_width)]
        for mlt_bin in range(effective_start, max_mlt, mlt_bin_width)
    }

    corr_dict = {}
    for lat_bin in range(0, n_lats, lat_bin_width):
        corr_dict[lat_bin] = {}
        for mlt_bin in range(effective_start, max_mlt, mlt_bin_width):
            mlt_iter = mlt_ranges[mlt_bin]
            v1 = ampere_stack[:, lat_bin:lat_bin + lat_bin_width, :][:, :, mlt_iter].ravel()
            v2 = prediction_stack[:, lat_bin:lat_bin + lat_bin_width, :][:, :, mlt_iter].ravel()
            corr_dict[lat_bin][mlt_bin] = np.corrcoef(v1, v2)[0, 1]
    return pd.DataFrame(corr_dict)

def creating_rmse_tables(results_dict, lat_bin_width=10, mlt_bin_width=3, mlt_bin_start=23):
    """
    Compute per-bin Pearson correlation between 'ampere' and 'predicted' across
    all timestamps in results_dict.

    Works at whatever resolution the DataFrames carry (50x24 for ACORN/BK,
    full Weimer resolution for Weimer).  Grid dimensions are read from the
    first entry rather than being hardcoded.
    """
    keys = list(results_dict.keys())
    if not keys:
        return pd.DataFrame()
    n_lats  = results_dict[keys[0]]['predicted'].shape[0]
    n_mlts  = results_dict[keys[0]]['predicted'].shape[1]

    # mlt_bin_start=23 was designed for the 24-column ACORN grid (wraps at 24).
    # For Weimer's finer MLT grid we start at 0 to cover all columns.
    effective_start = 0 if n_mlts > 24 else mlt_bin_start
    max_mlt = effective_start + n_mlts

    def _to_arr(x):
        return x.values if hasattr(x, 'values') else np.asarray(x)
    ampere_stack     = np.stack([_to_arr(results_dict[k]['ampere'])    for k in keys], axis=0)
    prediction_stack = np.stack([_to_arr(results_dict[k]['predicted']) for k in keys], axis=0)
    mlt_ranges = {
        mlt_bin: [(i % n_mlts) for i in range(mlt_bin, mlt_bin + mlt_bin_width)]
        for mlt_bin in range(effective_start, max_mlt, mlt_bin_width)
    }

    rmse_dict = {}
    for lat_bin in range(0, n_lats, lat_bin_width):
        rmse_dict[lat_bin] = {}
        for mlt_bin in range(effective_start, max_mlt, mlt_bin_width):
            mlt_iter = mlt_ranges[mlt_bin]
            v1 = ampere_stack[:, lat_bin:lat_bin + lat_bin_width, :][:, :, mlt_iter].ravel()
            v2 = prediction_stack[:, lat_bin:lat_bin + lat_bin_width, :][:, :, mlt_iter].ravel()
            rmse_dict[lat_bin][mlt_bin] = np.sqrt(MSE(v1, v2))/np.std(v1)
    return pd.DataFrame(rmse_dict)


def creating_mean_tables(results_dict, value, lat_bin_width=10, mlt_bin_width=3, mlt_bin_start=23):
    """
    Compute per-bin mean of a given value field across all timestamps.

    Works at whatever resolution the DataFrames carry (50x24 for ACORN/BK,
    full Weimer resolution for Weimer).  Grid dimensions are read from the
    first entry rather than being hardcoded.
    """
    keys = list(results_dict.keys())
    if not keys:
        return pd.DataFrame()
    n_lats  = results_dict[keys[0]][value].shape[0]
    n_mlts  = results_dict[keys[0]][value].shape[1]
    effective_start = 0 if n_mlts > 24 else mlt_bin_start
    max_mlt = effective_start + n_mlts

    stack = np.stack([results_dict[k][value].values for k in keys], axis=0)

    mlt_ranges = {
        mlt_bin: [(i % n_mlts) for i in range(mlt_bin, mlt_bin + mlt_bin_width)]
        for mlt_bin in range(effective_start, max_mlt, mlt_bin_width)
    }

    mean_dict = {}
    for lat_bin in range(0, n_lats, lat_bin_width):
        mean_dict[lat_bin] = {}
        for mlt_bin in range(effective_start, max_mlt, mlt_bin_width):
            mlt_iter = mlt_ranges[mlt_bin]
            mean_dict[lat_bin][mlt_bin] = stack[:, lat_bin:lat_bin + lat_bin_width, :][:, :, mlt_iter].mean()
    return pd.DataFrame(mean_dict)


## Temporal heatmaps

Keograms: MLAT against time at fixed MLT. These reveal the R1/R2 sheets as
bands and show them moving equatorward during a storm — structure that
individual polar snapshots do not convey.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Temporal heatmap helpers
# ═══════════════════════════════════════════════════════════════════════════
# Keograms: MLAT on the vertical axis, time on the horizontal, for a fixed
# MLT slice. These make the R1/R2 current sheets visible as bands and show
# how they move equatorward as a storm develops -- structure that polar
# snapshots at individual timesteps do not convey.
# ACORN physical MLATs: row 0 = 89°, row 49 = 40°
# Lat values depend on whether LAT_MASK_50 is active
n_lat_rows     = 40 if LAT_MASK_50 else 50
ACORN_MLAT_VALS = 90 - np.arange(1, n_lat_rows + 1).astype(float)
N_LATS = n_lat_rows

def apply_time_xticks(ax, timestamps, max_ticks=8):
    """
    Replace integer x-axis ticks with datetime labels drawn from timestamps.
    Selects up to max_ticks evenly-spaced positions and rotates labels 30°.
    """
    n = len(timestamps)
    step = max(1, n // max_ticks)
    tick_pos = list(range(0, n, step))
    tick_labels = [
        pd.Timestamp(timestamps[i]).strftime('%m-%d %H:%M')
        for i in tick_pos
    ]
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(tick_labels, rotation=30, ha='right', fontsize=8)


def plot_temporal_heatmap_at_mlt(
    results_dict, mlt_value, value_type='predicted',
    figsize=(14, 8), cmap='bwr', save_path=None
):
    timestamps = sorted(results_dict.keys())
    n_times = len(timestamps)
    data_array = np.zeros((N_LATS, n_times))
    for i, ts in enumerate(timestamps):
        data_array[:, i] = results_dict[ts][value_type].iloc[:, int(mlt_value)]

    # Build meshgrid: x = time index, y = MLAT (degrees)
    time_edges = np.arange(n_times + 1) - 0.5
    lat_edges  = np.concatenate([[90.5], ACORN_MLAT_VALS - 0.5])
    X, Y = np.meshgrid(time_edges, lat_edges)

    fig, ax = plt.subplots(figsize=figsize)
    vmax = np.max(np.abs(data_array)) if cmap == 'bwr' else np.max(data_array)
    vmin = -vmax if cmap == 'bwr' else np.min(data_array)
    im = ax.pcolormesh(X, Y, data_array, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_ylim(40, 90)
    apply_time_xticks(ax, timestamps)
    ax.set_xlabel('Time', fontsize=12, fontweight='bold')
    ax.set_ylabel('MLAT (°)', fontsize=12, fontweight='bold')
    ax.set_title(f'{value_type.capitalize()} at MLT={mlt_value}h Over Time',
                 fontsize=14, fontweight='bold')
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(r'FAC Density ($\mu$A/m²)' if value_type != 'std'
                   else r'Uncertainty ($\mu$A/m²)', fontsize=11)
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    return fig, ax


def plot_multi_mlt_temporal_heatmap(
    results_dict, mlt_values=[0, 6, 12, 18], value_type='predicted',
    figsize=(18, 12), cmap='bwr', save_path=None
):
    n_plots = len(mlt_values)
    n_cols  = 2
    n_rows  = int(np.ceil(n_plots / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = axes.flatten() if n_plots > 1 else [axes]

    timestamps = sorted(results_dict.keys())
    n_times = len(timestamps)

    # Build shared meshgrid and colour scale
    time_edges = np.arange(n_times + 1) - 0.5
    lat_edges  = np.concatenate([[90.5], ACORN_MLAT_VALS - 0.5])
    X, Y = np.meshgrid(time_edges, lat_edges)

    all_data = []
    for mlt in mlt_values:
        d = np.zeros((N_LATS, n_times))
        for i, ts in enumerate(timestamps):
            d[:, i] = results_dict[ts][value_type].iloc[:, int(mlt)]
        all_data.append(d)
    all_data = np.concatenate(all_data)
    vmax = np.max(np.abs(all_data)) if cmap == 'bwr' else np.max(all_data)
    vmin = -vmax if cmap == 'bwr' else np.min(all_data)

    for idx, (mlt, ax) in enumerate(zip(mlt_values, axes)):
        d = np.zeros((N_LATS, n_times))
        for i, ts in enumerate(timestamps):
            d[:, i] = results_dict[ts][value_type].iloc[:, int(mlt)]
        im = ax.pcolormesh(X, Y, d, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_ylim(40, 90)
        apply_time_xticks(ax, timestamps)
        ax.set_xlabel('Time', fontsize=10)
        ax.set_ylabel('MLAT (°)', fontsize=10)
        ax.set_title(f'MLT = {mlt}h', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

    for idx in range(n_plots, len(axes)):
        axes[idx].set_visible(False)
    fig.colorbar(im, ax=axes,
                 label=r'FAC Density ($\mu$A/m²)' if value_type != 'std'
                       else r'Uncertainty ($\mu$A/m²)',
                 orientation='vertical', fraction=0.046, pad=0.04)
    fig.suptitle(f'{value_type.capitalize()} Over Time at Different MLTs',
                 fontsize=16, fontweight='bold', y=0.995)
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    return fig, axes


## Generate temporal heatmaps

In [ ]:
may_acorn = {k: acorn_results[k] for k in may_keys if k in acorn_results}

fig, ax = plot_temporal_heatmap_at_mlt(
    may_acorn, mlt_value=0, value_type='predicted',
    save_path=f'plots/temporal_heatmap_mlt0_{RUN_TAG}.png'
)
plt.show()


In [ ]:
fig, axes = plot_multi_mlt_temporal_heatmap(
    may_acorn, mlt_values=[0, 6, 12, 18], value_type='predicted',
    save_path=f'plots/temporal_heatmap_multi_mlt_{RUN_TAG}.png'
)
plt.show()


In [ ]:
fig, axes = plot_multi_mlt_temporal_heatmap(
    may_acorn, mlt_values=[0, 6, 12, 18], value_type='std',
    cmap='Purples', save_path=f'plots/temporal_heatmap_uncertainty_{RUN_TAG}.png'
)
plt.show()


## AMPERE heatmap with input parameter overlay

Time-averaged maps with the SHAP regions outlined, so interpretability results
can be read against the structure they describe.

In [ ]:
# def plot_ampere_with_input(
#     results_dict, input_param_idx, input_param_name,
#     mlt_value=0, figsize=(14, 5), alpha_input=0.6,
#     save_path=None
# ):
#     """
#     Single-panel plot: AMPERE heatmap (MLAT vs time) at a given MLT with
#     the input parameter overlaid as a normalised line within the heatmap.
#     The heatmap is made semi-transparent so the line shows through.
#     The line is normalised to the MLAT range [40, 90] for display;
#     the legend shows the actual min/max values for scale reference.

#     Parameters
#     ----------
#     results_dict     : dict  — e.g. may_acorn
#     input_param_idx  : int   — index into the input sequence feature axis
#     input_param_name : str   — label for the title and legend
#     mlt_value        : int   — MLT column to slice for the heatmap
#     figsize          : tuple
#     alpha_input      : float — transparency of the input parameter line
#     save_path        : str or None
#     """
#     timestamps = sorted(results_dict.keys())
#     n_times    = len(timestamps)

#     def _arr(v, field):
#         x = v[field]
#         return x.values if hasattr(x, 'values') else np.asarray(x)

#     ampere_data = np.zeros((N_LATS, n_times))
#     input_vals  = np.zeros(n_times)

#     for i, ts in enumerate(timestamps):
#         v = results_dict[ts]
#         ampere_data[:, i] = _arr(v, 'ampere')[:, int(mlt_value)]

#         # Input sequence shape: (lookback, n_features) — take most recent timestep
#         inp = v['input']
#         inp = inp.values if hasattr(inp, 'values') else np.asarray(inp)
#         if inp.ndim == 2:
#             input_vals[i] = inp[-1, input_param_idx]
#         else:
#             input_vals[i] = inp[input_param_idx]

#     time_edges = np.arange(n_times + 1) - 0.5
#     lat_edges  = np.concatenate([[90.5], ACORN_MLAT_VALS - 0.5])
#     X, Y       = np.meshgrid(time_edges, lat_edges)

#     vmax = np.max(np.abs(ampere_data))
#     norm = mpl.colors.Normalize(vmin=-vmax, vmax=vmax)

#     # ── Plot ──────────────────────────────────────────────────────────────
#     fig, ax1 = plt.subplots(figsize=figsize)

#     # Normalise input values into [40, 90] MLAT range for overlay.
#     # Avoids twinx inverted-axis bleed-through issues.
#     # Note: axis is inverted so 90 is at top — high input values map to top.
#     inp_norm  = input_vals - input_vals.min()
#     inp_range = input_vals.max() - input_vals.min()
#     if inp_range > 0:
#         inp_scaled = 40 + (inp_norm / inp_range) * 50
#     else:
#         inp_scaled = np.full_like(input_vals, 65.0)

#     ax1.plot(range(n_times), inp_scaled,
#              color='black', lw=1.5, alpha=alpha_input,
#              label=input_param_name, zorder=4)

#         # AMPERE heatmap — semi-transparent so input line shows through
#     im = ax1.pcolormesh(X, Y, ampere_data, cmap='bwr', norm=norm, zorder=2)
#     im.set_alpha(0.75)
#     ax1.set_ylim(40, 90)
#     # ax1.invert_yaxis()
#     ax1.set_ylabel('MLAT (°)', fontsize=11)
#     ax1.set_title(f'AMPERE at MLT={mlt_value}h  |  {input_param_name}',
#                   fontsize=13, fontweight='bold')
#     ax1.grid(True, alpha=1, linestyle='--', zorder=3)
#     apply_time_xticks(ax1, timestamps)
#     ax1.set_xlabel('Time', fontsize=11)
#     fig.colorbar(im, ax=ax1, label=r'FAC ($\mu$A/m²)', fraction=0.02, pad=0.01)

#     ax1.legend(
#         [plt.Line2D([0], [0], color='black', lw=1.5)],
#         [f'{input_param_name}  [{input_vals.min():.2f} – {input_vals.max():.2f}]'],
#         loc='upper right', fontsize=9, framealpha=0.7
#     )

#     if save_path:
#         os.makedirs(os.path.dirname(save_path), exist_ok=True)
#         plt.savefig(save_path, dpi=150, bbox_inches='tight')
#     plt.show()
#     return fig, ax1


# # ── Generate one plot per input parameter (May storm) ────────────────────
# may_acorn = {k: acorn_results[k] for k in may_keys if k in acorn_results}

# for _idx, _param in enumerate(SCI_CONFIG['input_params']):
#     plot_ampere_with_input(
#         may_acorn,
#         input_param_idx=_idx,
#         input_param_name=_param,
#         mlt_value=9,
#         alpha_input=0.2,
#         save_path=f'plots/model_results/ampere_input_{_param.replace(" ", "_")}_{RUN_TAG}.png'
#     )

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Mean polar maps with SHAP region overlays
# ═══════════════════════════════════════════════════════════════════════════
# Time-averaged FAC maps with the SHAP analysis regions outlined on top, so
# the interpretability results can be read against the physical structure
# they describe.
#
# REGIONS below is the same 12-region decomposition used in shap_values.py:
# three MLAT bands (auroral / R1 / R2) x four MLT sectors (day, dusk, night,
# dawn). Nightside sectors wrap through midnight, hence mlt_start > mlt_end.
# ── Mean polar maps with SHAP region overlays ────────────────────────────
import matplotlib.patheffects as pe

REGIONS = [
    {'mlat_low': 80.0, 'mlat_high': 90.0, 'mlt_start':  9, 'mlt_end': 14, 'label': 'Dayside Auroral'},
    {'mlat_low': 80.0, 'mlat_high': 90.0, 'mlt_start': 15, 'mlt_end': 20, 'label': 'Dusk Auroral'},
    {'mlat_low': 80.0, 'mlat_high': 90.0, 'mlt_start': 21, 'mlt_end':  2, 'label': 'Nightside Auroral'},
    {'mlat_low': 80.0, 'mlat_high': 90.0, 'mlt_start':  3, 'mlt_end':  8, 'label': 'Dawn Auroral'},
    {'mlat_low': 70.0, 'mlat_high': 79.0, 'mlt_start':  9, 'mlt_end': 14, 'label': 'Dayside R1'},
    {'mlat_low': 70.0, 'mlat_high': 79.0, 'mlt_start': 15, 'mlt_end': 20, 'label': 'Dusk R1'},
    {'mlat_low': 70.0, 'mlat_high': 79.0, 'mlt_start': 21, 'mlt_end':  2, 'label': 'Nightside R1'},
    {'mlat_low': 70.0, 'mlat_high': 79.0, 'mlt_start':  3, 'mlt_end':  8, 'label': 'Dawn R1'},
    {'mlat_low': 50.0, 'mlat_high': 69.0, 'mlt_start':  9, 'mlt_end': 14, 'label': 'Dayside R2'},
    {'mlat_low': 50.0, 'mlat_high': 69.0, 'mlt_start': 15, 'mlt_end': 20, 'label': 'Dusk R2'},
    {'mlat_low': 50.0, 'mlat_high': 69.0, 'mlt_start': 21, 'mlt_end':  2, 'label': 'Nightside R2'},
    {'mlat_low': 50.0, 'mlat_high': 69.0, 'mlt_start':  3, 'mlt_end':  8, 'label': 'Dawn R2'},
]

REGION_COLORS = {
    (80.0, 90.0): ('green',  0.10),
    (70.0, 79.0): ('yellow', 0.10),
    (50.0, 69.0): ('orange',   0.05),
}


def draw_region_fill(ax, mlat_low, mlat_high, mlt_start, mlt_end, color, alpha, n_rad, n_pts=200):
    """
    Shade a SHAP region and draw solid boundary lines on all four edges.
    Right edge is inclusive (extended by 1h bin).
    Handles midnight-crossing sectors.
    """
    r_inner = 90.0 - mlat_high
    r_outer = 90.0 - mlat_low
    mlt_end_inc = mlt_end + 1

    def mlt_to_theta(mlt):
        return (mlt / 24.0) * 2 * np.pi

    th_start = mlt_to_theta(mlt_start)
    th_end   = mlt_to_theta(mlt_end_inc)
    if mlt_end < mlt_start:
        th_end += 2 * np.pi

    th_arc = np.linspace(th_start, th_end, n_pts)

    # Shaded fill
    ax.fill_between(th_arc, r_inner, r_outer,
                    color=color, alpha=alpha, zorder=2, linewidth=0)

    # Boundary lines
    bkw = dict(color=color, lw=1.5, zorder=4, solid_capstyle='round')

    # Inner and outer arcs
    ax.plot(th_arc, np.full(n_pts, r_inner), **bkw)
    ax.plot(th_arc, np.full(n_pts, r_outer), **bkw)

    # Left and right radial edges
    ax.plot([th_start, th_start], [r_inner, r_outer], **bkw)
    ax.plot([th_end,   th_end],   [r_inner, r_outer], **bkw)


# ── Compute mean tables ───────────────────────────────────────────────────
may_acorn_results = {k: acorn_results[k] for k in may_keys if k in acorn_results}

panels = [
    (may_acorn_results, 'ampere',    'AMPERE\n(Storm)'),
    (may_acorn_results, 'predicted', 'ACORN Sci\n(Storm)'),
    (acorn_results,     'ampere',    'AMPERE\n(All)'),
    (acorn_results,     'predicted', 'ACORN Sci\n(All)'),
]

mean_tables = [
    creating_mean_tables(rd, value=v, mlt_bin_width=1, lat_bin_width=1)
    for rd, v, _ in panels
]

vmax   = max(np.abs(t.values).max() for t in mean_tables)
norm   = mpl.colors.Normalize(vmin=-vmax, vmax=vmax)
n_rad = 40 if LAT_MASK_50 else 50

theta_ticks = np.linspace(0, 2*np.pi, 8, endpoint=False)
rad_step   = max(1, n_rad // 5)
rad_ticks   = list(range(0, n_rad, rad_step))
rad_labels  = [str(int(89 - t)) for t in rad_ticks]

fig, axs = plt.subplots(1, 4, figsize=(22, 6), subplot_kw=dict(projection='polar'))
fig.suptitle('Mean FAC — Storm Window vs Full Test Set\n(shaded = SHAP regions)',
             fontsize=15, fontweight='bold', y=1.02)

for ax, (_, _, title), table in zip(axs, panels, mean_tables):
    n_mlt_bins = len(table)
    n_lat_bins = len(table.columns)
    r_p, th_p  = np.meshgrid(
        np.linspace(0, n_rad, n_lat_bins, endpoint=False),
        np.linspace(0, 2*np.pi, n_mlt_bins, endpoint=False)
    )
    ax.pcolormesh(th_p, r_p, table, cmap='bwr', norm=norm, zorder=1)
    polar_setup(ax, n_rad, rad_ticks=rad_ticks, rad_labels=rad_labels,
                mlt_fontsize=9, rad_fontsize=8, grid_alpha=0.2, grid_zorder=3)
    ax.set_title(title, fontsize=12)
    ax.set_ylim(0, n_rad)

    # for region in REGIONS:
    #     color, alpha = REGION_COLORS[(region['mlat_low'], region['mlat_high'])]
    #     draw_region_fill(
    #         ax,
    #         mlat_low=region['mlat_low'], mlat_high=region['mlat_high'],
    #         mlt_start=region['mlt_start'], mlt_end=region['mlt_end'],
    #         color=color, alpha=alpha, n_rad=n_rad,
    #     )

# ── Legend ────────────────────────────────────────────────────────────────
# legend_elements = [
#     mpl.patches.Patch(facecolor='green',  alpha=0.5, label='Auroral (80–90°)'),
#     mpl.patches.Patch(facecolor='yellow', alpha=0.5, label='Region 1 (70–79°)'),
#     mpl.patches.Patch(facecolor='orange',   alpha=0.5, label='Region 2 (50–69°)'),
# ]
# fig.legend(handles=legend_elements, loc='lower center', ncol=3,
#            fontsize=10, framealpha=0.5, bbox_to_anchor=(0.5, -0.05))

c = axs[-1].collections[0]
fig.colorbar(c, ax=axs.ravel().tolist(),
             label=r'Mean FAC ($\mu$A/m²)', orientation='vertical',
             fraction=0.02, pad=0.04)
plt.show()

## Compute correlations and RMSE

Metrics on both the full test set and the Weimer window. The two are **not
comparable** — the storm window is far more active than the test set average.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Restrict every model to the Weimer window
# ═══════════════════════════════════════════════════════════════════════════
# Weimer output covers only the May 2023 storm, so a fair five-way comparison
# has to be made on the intersection of all key sets. The unrestricted
# versions are kept alongside so full-test-set metrics remain available --
# metrics computed on the storm window alone are not comparable to those over
# the whole test set, since the storm window is far more active.
# Weimer-window subsets
acorn_results_w = {k: v for k, v in acorn_results.items() if k in weimer_keys}
opp_results_w   = {k: v for k, v in op_results.items()   if k in weimer_keys}
bk_sci_results_w = {k: v for k, v in bk_sci_results.items() if k in weimer_keys}
bk_opp_results_w = {k: v for k, v in bk_op_results.items() if k in weimer_keys}

# ── Figure A: Full test set correlation and RMSE ──────────────────────────
opp_corr     = creating_correlation_tables(op_results,     mlt_bin_width=1, lat_bin_width=1)
sci_corr     = creating_correlation_tables(acorn_results,   mlt_bin_width=1, lat_bin_width=1)
bk_sci_corr  = creating_correlation_tables(bk_sci_results,  mlt_bin_width=1, lat_bin_width=1)
bk_opp_corr  = creating_correlation_tables(bk_op_results,  mlt_bin_width=1, lat_bin_width=1)

opp_rmse     = creating_rmse_tables(op_results,     mlt_bin_width=1, lat_bin_width=1)
sci_rmse     = creating_rmse_tables(acorn_results,   mlt_bin_width=1, lat_bin_width=1)
bk_sci_rmse  = creating_rmse_tables(bk_sci_results,  mlt_bin_width=1, lat_bin_width=1)
bk_opp_rmse  = creating_rmse_tables(bk_op_results,  mlt_bin_width=1, lat_bin_width=1)

# ── Figure B: Weimer-window metrics ──────────────────────────────────────
opp_corr_w     = creating_correlation_tables(opp_results_w,     mlt_bin_width=1, lat_bin_width=1)
sci_corr_w     = creating_correlation_tables(acorn_results_w,   mlt_bin_width=1, lat_bin_width=1)
bk_sci_corr_w  = creating_correlation_tables(bk_sci_results_w,  mlt_bin_width=1, lat_bin_width=1)
bk_opp_corr_w  = creating_correlation_tables(bk_opp_results_w,  mlt_bin_width=1, lat_bin_width=1)
weimer_corr    = creating_correlation_tables(weimer_results,     mlt_bin_width=1, lat_bin_width=1)

opp_rmse_w     = creating_rmse_tables(opp_results_w,     mlt_bin_width=1, lat_bin_width=1)
sci_rmse_w     = creating_rmse_tables(acorn_results_w,   mlt_bin_width=1, lat_bin_width=1)
bk_sci_rmse_w  = creating_rmse_tables(bk_sci_results_w,  mlt_bin_width=1, lat_bin_width=1)
bk_opp_rmse_w  = creating_rmse_tables(bk_opp_results_w,  mlt_bin_width=1, lat_bin_width=1)
weimer_rmse    = creating_rmse_tables(weimer_results,     mlt_bin_width=1, lat_bin_width=1)


## Polar correlation plots

In [ ]:
theta_ticks = np.linspace(0, 2*np.pi, 8, endpoint=False)
n_rad      = 40 if LAT_MASK_50 else 50
rad_div     = 4 if LAT_MASK_50 else 4
rad_step   = max(1, n_rad // rad_div)
rad_ticks   = list(range(0, n_rad, rad_step))
rad_labels  = [str(int(90 - t)) for t in rad_ticks]

def plot_corr_polar(tables_titles, suptitle, save_path):
    all_tables = [t for t, _ in tables_titles]
    scale_max  = max(c.max().max() for c in all_tables)
    scale_min  = min(c.min().min() for c in all_tables)
    scale_map  = mpl.colors.Normalize(vmin=scale_min, vmax=scale_max)
    n = len(tables_titles)
    fig, axs = plt.subplots(ncols=n, nrows=1, figsize=(6*n, 6), subplot_kw=dict(projection='polar'))
    plt.suptitle(suptitle, fontsize=20)
    for ax, (corr_table, title) in zip(axs, tables_titles):
        n_mlt_bins = len(corr_table)
        n_lat_bins = len(corr_table.columns)
        r_p, th_p  = np.meshgrid(
            np.linspace(0, n_rad, n_lat_bins, endpoint=False),
            np.linspace(0, 2*np.pi, n_mlt_bins, endpoint=False)
        )
        ax.set_title(title, fontsize=12)
        c = ax.pcolormesh(th_p, r_p, corr_table, cmap='viridis', norm=scale_map)
        polar_setup(ax, n_rad, rad_ticks=rad_ticks, rad_labels=rad_labels,
                    grid_alpha=0.5, ylim=n_rad - 1)
    fig.colorbar(c, ax=axs.ravel().tolist(), label='Corr Coefficient', orientation='vertical')
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_corr_polar([
    (opp_corr,    'ACORN Op (all)'),
    (sci_corr,    'ACORN Sci (all)'),
    (bk_sci_corr, 'BK Sci (all)'),
    (bk_opp_corr, 'BK Op (all)'),
], 'Correlation Coefficients — Full Test Set',
   f'plots/model_results/corr_all_{RUN_TAG}.png')

plot_corr_polar([
    (opp_corr_w,    'ACORN Op (Weimer window)'),
    (sci_corr_w,    'ACORN Sci (Weimer window)'),
    (bk_sci_corr_w, 'BK Sci (Weimer window)'),
    (bk_opp_corr_w, 'BK Op (Weimer window)'),
    (weimer_corr,   'Weimer'),
], 'Correlation Coefficients — Weimer Window',
   f'plots/model_results/corr_weimer_{RUN_TAG}.png')


## Polar RMSE plots

In [ ]:
theta_ticks = np.linspace(0, 2*np.pi, 8, endpoint=False)
n_rad      = 40 if LAT_MASK_50 else 50
rad_div     = 5 if LAT_MASK_50 else 4
rad_step   = max(1, n_rad // rad_div)
rad_ticks   = list(range(0, n_rad, rad_step))
rad_labels  = [str(int(89 - t)) for t in rad_ticks]

def plot_rmse_polar(tables_titles, suptitle, save_path):
    all_tables = [t for t, _ in tables_titles]
    scale_max  = max(c.max().max() for c in all_tables)
    scale_min  = min(c.min().min() for c in all_tables)
    scale_map  = mpl.colors.Normalize(vmin=scale_min, vmax=scale_min*3)
    n = len(tables_titles)
    fig, axs = plt.subplots(ncols=n, nrows=1, figsize=(6*n, 6), subplot_kw=dict(projection='polar'))
    plt.suptitle(suptitle, fontsize=20)
    for ax, (rmse_table, title) in zip(axs, tables_titles):
        n_mlt_bins = len(rmse_table)
        n_lat_bins = len(rmse_table.columns)
        r_p, th_p  = np.meshgrid(
            np.linspace(0, n_rad, n_lat_bins, endpoint=False),
            np.linspace(0, 2*np.pi, n_mlt_bins, endpoint=False)
        )
        ax.set_title(title, fontsize=12)
        c = ax.pcolormesh(th_p, r_p, rmse_table, cmap='viridis', norm=scale_map)
        polar_setup(ax, n_rad, rad_ticks=rad_ticks, rad_labels=rad_labels,
                    grid_alpha=0.5, ylim=n_rad - 1)
    fig.colorbar(c, ax=axs.ravel().tolist(), label=r'RMSE ($\mu A/m^{2}$)', orientation='vertical')
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_rmse_polar([
    (opp_rmse,    'ACORN Op (all)'),
    (sci_rmse,    'ACORN Sci (all)'),
    (bk_sci_rmse, 'Kunduri Sci (all)'),
    (bk_opp_rmse, 'Kunduri Op (all)'),
], 'Root-Mean Square Error — Full Test Set',
   f'plots/model_results/rmse_all_{RUN_TAG}.png')

plot_rmse_polar([
    (opp_rmse_w,    'ACORN Op (Weimer window)'),
    (sci_rmse_w,    'ACORN Sci (Weimer window)'),
    (bk_sci_rmse_w, 'Kunduri Sci (Weimer window)'),
    (bk_opp_rmse_w, 'Kunduri Op (Weimer window)'),
    (weimer_rmse,   'Weimer'),
], 'Root-Mean Square Error — Weimer Window',
   f'plots/model_results/rmse_weimer_{RUN_TAG}.png')


## Regional metrics — HSS, normalised RMSE, AUC-PR

Three metrics probing different failure modes, over 12 regions: three MLAT
bands (auroral, R1, R2) × four MLT sectors.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Regional skill metrics: HSS, normalised RMSE, AUC-PR
# ═══════════════════════════════════════════════════════════════════════════
# Correlation alone rewards getting the pattern right while missing the
# amplitude. These three probe different failure modes:
#
#   HSS     -- skill at detecting significant current above a threshold,
#              scored against random chance. Sensitive to timing.
#   nRMSE   -- amplitude error normalised by the observed range, so regions
#              of very different magnitude are comparable.
from sklearn.metrics import precision_recall_curve, auc as sklearn_auc

# ── Region definitions ────────────────────────────────────────────────────
REGIONS = [
    {'mlat_low': 80.0, 'mlat_high': 90.0, 'mlt_start':  9, 'mlt_end': 14, 'label': 'R0 Dayside'},
    {'mlat_low': 80.0, 'mlat_high': 90.0, 'mlt_start': 15, 'mlt_end': 20, 'label': 'R0 Dusk'},
    {'mlat_low': 80.0, 'mlat_high': 90.0, 'mlt_start': 21, 'mlt_end':  2, 'label': 'R0 Nightside'},
    {'mlat_low': 80.0, 'mlat_high': 90.0, 'mlt_start':  3, 'mlt_end':  8, 'label': 'R0 Dawn'},
    {'mlat_low': 70.0, 'mlat_high': 79.0, 'mlt_start':  9, 'mlt_end': 14, 'label': 'R1 Dayside'},
    {'mlat_low': 70.0, 'mlat_high': 79.0, 'mlt_start': 15, 'mlt_end': 20, 'label': 'R1 Dusk'},
    {'mlat_low': 70.0, 'mlat_high': 79.0, 'mlt_start': 21, 'mlt_end':  2, 'label': 'R1 Nightside'},
    {'mlat_low': 70.0, 'mlat_high': 79.0, 'mlt_start':  3, 'mlt_end':  8, 'label': 'R1 Dawn'},
    {'mlat_low': 50.0, 'mlat_high': 69.0, 'mlt_start':  9, 'mlt_end': 14, 'label': 'R2 Dayside'},
    {'mlat_low': 50.0, 'mlat_high': 69.0, 'mlt_start': 15, 'mlt_end': 20, 'label': 'R2 Dusk'},
    {'mlat_low': 50.0, 'mlat_high': 69.0, 'mlt_start': 21, 'mlt_end':  2, 'label': 'R2 Nightside'},
    {'mlat_low': 50.0, 'mlat_high': 69.0, 'mlt_start':  3, 'mlt_end':  8, 'label': 'R2 Dawn'},
]

# ── Config ────────────────────────────────────────────────────────────────
HSS_WINDOW_MINUTES = 10
POLAR_YLIM         = 40

# ── Index helpers ─────────────────────────────────────────────────────────
_MLAT_MIN, _MLAT_MAX, _N_MLAT, _N_MLT = 40.0, 90.0, 50, 24

def mlat_to_idx(mlat_low, mlat_high):
    bw       = (_MLAT_MAX - _MLAT_MIN) / _N_MLAT
    idx_pole = int((_MLAT_MAX - mlat_high) / bw)
    idx_eq   = min(int((_MLAT_MAX - mlat_low) / bw), _N_MLAT - 1)
    return list(range(idx_pole, idx_eq + 1))

def mlt_to_idx(mlt_start, mlt_end):
    s, e = int(mlt_start) % _N_MLT, int(mlt_end) % _N_MLT
    return list(range(s, _N_MLT)) + list(range(0, e + 1)) if s > e else list(range(s, e + 1))


# ── Step 1: pre-stack ─────────────────────────────────────────────────────
def stack_dict(results_dict):
    def _arr(v, field):
        x = v[field]
        return x.values if hasattr(x, 'values') else np.asarray(x)
    keys = list(results_dict.keys())
    obs  = np.stack([_arr(results_dict[k], 'ampere')    for k in keys], axis=0)
    pred = np.stack([_arr(results_dict[k], 'predicted') for k in keys], axis=0)
    return obs, pred, keys

def stack_dict_weimer(results_dict):
    def _arr(v, field):
        x = v[field]
        return x.values if hasattr(x, 'values') else np.asarray(x)
    keys = [k for k in results_dict if np.any(_arr(results_dict[k], 'ampere') != 0)]
    print(f'  Weimer: {len(keys)}/{len(results_dict)} timestamps have valid AMPERE')
    obs  = np.stack([_arr(results_dict[k], 'ampere')    for k in keys], axis=0)
    pred = np.stack([_arr(results_dict[k], 'predicted') for k in keys], axis=0)
    return obs, pred, keys

def region_idx(mlat_low, mlat_high, mlt_start, mlt_end):
    lat_idx = mlat_to_idx(mlat_low, mlat_high)
    mlt_idx = mlt_to_idx(mlt_start, mlt_end)
    max_row = 40 if LAT_MASK_50 else 50
    return [i for i in lat_idx if i < max_row], mlt_idx

def extract_region_means(obs_stack, pred_stack, mlat_low, mlat_high, mlt_start, mlt_end):
    lat_idx, mlt_idx = region_idx(mlat_low, mlat_high, mlt_start, mlt_end)
    if not lat_idx:
        return np.array([]), np.array([])
    obs_means  = np.abs(obs_stack [:, :, :][:, lat_idx, :][:, :, mlt_idx]).mean(axis=(1, 2))
    pred_means = np.abs(pred_stack[:, :, :][:, lat_idx, :][:, :, mlt_idx]).mean(axis=(1, 2))
    return obs_means, pred_means


# ── Step 2: metric functions ──────────────────────────────────────────────
def compute_hss(obs_block, pred_block, threshold):
    o = obs_block  >= threshold
    p = pred_block >= threshold
    a = np.sum( o &  p)
    b = np.sum(~o &  p)
    c = np.sum( o & ~p)
    d = np.sum(~o & ~p)
    n = a + b + c + d
    if n == 0:
        return np.nan
    expected = ((a + c) * (a + b) + (b + d) * (c + d)) / n
    denom    = n - expected
    return (a + d - expected) / denom if denom != 0 else np.nan

def compute_nrmse(obs_means, pred_means):
    if len(obs_means) < 2:
        return np.nan
    rmse = np.sqrt(np.mean((obs_means - pred_means) ** 2))
    std  = np.std(obs_means)
    return rmse / std if std > 0 else np.nan


# ── Step 3: model dicts ───────────────────────────────────────────────────
MODEL_DICTS_ALL = {
    'ACORN Op': op_results,
    'ACORN Sci': acorn_results,
    'Kunduri Op':    bk_op_results,
    'Kunduri Sci':    bk_sci_results,
}
OBS_SOURCE_ALL = {
    'ACORN Op': acorn_results,
    'ACORN Sci': acorn_results,
    'Kunduri Op':    acorn_results,
    'Kunduri Sci':    acorn_results,
}
MODEL_DICTS_W = {
    'ACORN Op': opp_results_w,
    'ACORN Sci': acorn_results_w,
    'Kunduri Op':    bk_opp_results_w,
    'Kunduri Sci':    bk_sci_results_w,
    'Weimer':    weimer_results,
}
OBS_SOURCE_W = {
    'ACORN Op': acorn_results_w,
    'ACORN Sci': acorn_results_w,
    'Kunduri Op':    acorn_results_w,
    'Kunduri Sci':    acorn_results_w,
    'Weimer':    weimer_results,
}
MODEL_DICTS = {**MODEL_DICTS_ALL, **MODEL_DICTS_W}
OBS_SOURCE  = {**OBS_SOURCE_ALL,  **OBS_SOURCE_W}

PERCENTILES = {'p50': 50, 'p75': 75, 'p90': 90, 'p99': 99}

print('Pre-stacking arrays...')
all_named = {
    'acorn_results':    acorn_results,
    'op_results':      op_results,
    'bk_sci_results':   bk_sci_results,
    'bk_op_results':   bk_op_results,
    'acorn_results_w':  acorn_results_w,
    'opp_results_w':    opp_results_w,
    'bk_sci_results_w': bk_sci_results_w,
    'bk_opp_results_w': bk_opp_results_w,
}
stacked = {}
for name, rd in all_named.items():
    stacked[name] = stack_dict(rd)
    print(f'  Stacked {name}: {stacked[name][0].shape}')
stacked['weimer_results'] = stack_dict_weimer(weimer_results)
print(f'  Stacked weimer_results: {stacked["weimer_results"][0].shape}')

PRED_KEY = {
    'ACORN Op': 'op_results',
    'ACORN Sci': 'acorn_results',
    'Kunduri Op':    'bk_op_results',
    'Kunduri Sci':    'bk_sci_results',
    'ACORN Op': 'opp_results_w',
    'ACORN Sci': 'acorn_results_w',
    'Kunduri Op':    'bk_opp_results_w',
    'Kunduri Sci':    'bk_sci_results_w',
    'Weimer':    'weimer_results',
}
OBS_KEY = {
    'ACORN Op': 'acorn_results',
    'ACORN Sci': 'acorn_results',
    'Kunduri Op':    'acorn_results',
    'Kunduri Sci':    'acorn_results',
    'ACORN Op': 'acorn_results_w',
    'ACORN Sci': 'acorn_results_w',
    'Kunduri Op':    'acorn_results_w',
    'Kunduri Sci':    'acorn_results_w',
    'Weimer':    'weimer_results',
}

# ── Universal thresholds ──────────────────────────────────────────────────
print('Computing universal thresholds...')
obs_stack_full, _, obs_keys_full = stacked['acorn_results']
universal_thresholds = {}
for region in REGIONS:
    rkw_full = dict(
        mlat_low=region['mlat_low'], mlat_high=region['mlat_high'],
        mlt_start=region['mlt_start'], mlt_end=region['mlt_end'],
    )
    obs_means_full, _ = extract_region_means(obs_stack_full, obs_stack_full, **rkw_full)
    obs_idx_full      = pd.to_datetime(obs_keys_full)
    obs_block_full    = (pd.Series(obs_means_full, index=obs_idx_full)
                         .resample(f'{HSS_WINDOW_MINUTES}min').mean()
                         .dropna().values)
    universal_thresholds[region['label']] = {
        pkey: np.percentile(obs_block_full, pct) if len(obs_block_full) > 0 else np.nan
        for pkey, pct in PERCENTILES.items()
    }
print('  Done.')

# ── Compute HSS / AUC-PR + full 50x24 NRMSE and Corr tables ──────────────
metrics     = {m: {} for m in MODEL_DICTS}
rmse_tables = {}
corr_tables = {}

print('Computing metrics and full tables...')
for model_label in MODEL_DICTS:
    print(f'  {model_label}...')
    obs_stack,  _, obs_keys  = stacked[OBS_KEY[model_label]]
    _, pred_stack, pred_keys = stacked[PRED_KEY[model_label]]

    # Full 50x24 tables
    rmse_tables[model_label] = creating_rmse_tables(
        MODEL_DICTS[model_label], mlt_bin_width=1, lat_bin_width=1)
    corr_tables[model_label] = creating_correlation_tables(
        MODEL_DICTS[model_label], mlt_bin_width=1, lat_bin_width=1)

    for region in REGIONS:
        rkw = dict(
            mlat_low=region['mlat_low'], mlat_high=region['mlat_high'],
            mlt_start=region['mlt_start'], mlt_end=region['mlt_end'],
        )
        obs_means, pred_means = extract_region_means(obs_stack, pred_stack, **rkw)

        obs_block  = (pd.Series(obs_means,  index=pd.to_datetime(obs_keys))
                      .resample(f'{HSS_WINDOW_MINUTES}min').mean().dropna().values)
        pred_block = (pd.Series(pred_means, index=pd.to_datetime(pred_keys))
                      .resample(f'{HSS_WINDOW_MINUTES}min').mean().dropna().values)
        n = min(len(obs_block), len(pred_block))
        obs_block, pred_block = obs_block[:n], pred_block[:n]

        entry = {}
        for pkey in PERCENTILES:
            thresh               = universal_thresholds[region['label']][pkey]
            entry[f'hss_{pkey}'] = compute_hss(obs_block, pred_block, thresh)
        metrics[model_label][region['label']] = entry

print('Done.')

In [ ]:
# ── Step 4: panel helpers ─────────────────────────────────────────────────
def setup_ax(ax, n_rad, theta_ticks, rad_ticks, rad_labels, row, col, title, row_label):
    if row_label=='Norm RMSE':
        rad_color='black'
    else:
        rad_color='black'
    polar_setup(ax, n_rad, rad_ticks=rad_ticks, rad_labels=rad_labels,
                mlt_fontsize=14, rad_fontsize=12, rad_color=rad_color,
                grid_alpha=0.3, grid_zorder=3, ylim=POLAR_YLIM)
    ax.set_facecolor('#1a1a1a')
    ax.spines['polar'].set_visible(False)
    if row == 0:
        ax.set_title(title, fontsize=25)
    if col == 0:
        ax.set_ylabel(row_label, fontsize=25, labelpad=15)


def draw_polar_panel(ax, table, cmap, norm, n_rad, theta_ticks, rad_ticks, rad_labels,
                      row=0, col=0, title='', row_label=''):
    """Full resolution table as pcolormesh."""
    setup_ax(ax, n_rad, theta_ticks, rad_ticks, rad_labels, row, col, title, row_label)
    n_mlt_bins = len(table)
    n_lat_bins = len(table.columns)
    r_p, th_p  = np.meshgrid(
        np.linspace(0, n_rad, n_lat_bins, endpoint=False),
        np.linspace(0, 2*np.pi, n_mlt_bins, endpoint=False)
    )
    ax.pcolormesh(th_p, r_p, table, cmap=cmap, norm=norm)


def draw_sector_panel(ax, region_vals, cmap, norm, n_rad,
                       theta_ticks, rad_ticks, rad_labels,
                       row=0, col=0, title='', row_label=''):
    """Sector fills for HSS panels."""
    setup_ax(ax, n_rad, theta_ticks, rad_ticks, rad_labels, row, col, title, row_label)
    n_pts = 200
    for region in REGIONS:
        val = region_vals.get(region['label'], np.nan)
        if not np.isfinite(val):
            continue
        r_inner     = 90.0 - region['mlat_high']
        r_outer     = 90.0 - region['mlat_low']
        mlt_end_inc = region['mlt_end'] + 1
        th_start    = (region['mlt_start'] / 24.0) * 2 * np.pi
        th_end      = (mlt_end_inc          / 24.0) * 2 * np.pi
        if region['mlt_end'] < region['mlt_start']:
            th_end += 2 * np.pi
        th_arc = np.linspace(th_start, th_end, n_pts)
        color  = mpl.cm.get_cmap(cmap)(norm(val))
        ax.fill_between(th_arc, r_inner, r_outer, color=color, zorder=2, linewidth=0)
        bkw = dict(color='white', lw=1.0, zorder=4, alpha=0.6)
        ax.plot(th_arc, np.full(n_pts, r_inner), **bkw)
        ax.plot(th_arc, np.full(n_pts, r_outer), **bkw)
        ax.plot([th_start,            th_start           ], [r_inner, r_outer], **bkw)
        ax.plot([th_end % (2*np.pi),  th_end % (2*np.pi) ], [r_inner, r_outer], **bkw)
        th_mid = (th_start + th_end) / 2
        r_mid  = (r_inner  + r_outer) / 2
        ax.text(th_mid, r_mid, f'{val:.2f}',
                ha='center', va='center', fontsize=10, fontweight='bold',
                color='white', zorder=5)


# ── Step 5: combined figure ───────────────────────────────────────────────
def plot_combined_metrics_grid(
    metrics, rmse_tables, corr_tables,
    hss_keys=['hss_p50', 'hss_p75', 'hss_p90', 'hss_p99'],
    model_labels=None, figsize_per_panel=(5, 5),
    hspace=0.05, wspace=0.3, top=0.94, bottom=0.02, right=0.88,
    save_path=None, title=None
):
    """
    Combined figure: N HSS rows (sector fills, magma) +
    NRMSE row (full 50x24, magma) + Corr row (full 50x24, bwr symmetric).
    Three separate colourbars on the right.
    """
    model_labels = model_labels or list(MODEL_DICTS.keys())
    n_cols = len(model_labels)
    n_hss  = len(hss_keys)
    n_rows = n_hss + 2
    n_rad = 40 if LAT_MASK_50 else 50

    theta_ticks = np.linspace(0, 2*np.pi, 8, endpoint=False)
    rad_step   = max(1, n_rad // 5)
    rad_ticks   = list(range(0, n_rad, rad_step))
    rad_labels  = [str(int(90 - t)) for t in rad_ticks]

    fig, axs = plt.subplots(
        n_rows, n_cols,
        figsize=(figsize_per_panel[0] * n_cols, figsize_per_panel[1] * n_rows),
        subplot_kw=dict(projection='polar')
    )
    fig.subplots_adjust(hspace=hspace, wspace=wspace, top=top, bottom=bottom, right=right)
    fig.suptitle(f'Regional Model Metrics - {title}', fontsize=35, fontweight='bold', y=top + 0.01)

    # ── Colour scales ─────────────────────────────────────────────────────
    hss_vals = [metrics[m][r['label']][hk]
                for hk in hss_keys for m in model_labels for r in REGIONS]
    hss_norm = mpl.colors.Normalize(
        vmin=min(v for v in hss_vals if np.isfinite(v)),
        vmax=max(v for v in hss_vals if np.isfinite(v)))

    nrmse_vals = [v for m in model_labels
                  for v in rmse_tables[m].values.ravel() if np.isfinite(v)]
    nrmse_norm = mpl.colors.Normalize(vmin=min(nrmse_vals), vmax=max(nrmse_vals))

    corr_vals  = [v for m in model_labels
                  for v in corr_tables[m].values.ravel() if np.isfinite(v)]
    corr_vmax  = max(abs(min(corr_vals)), abs(max(corr_vals)))
    corr_norm  = mpl.colors.Normalize(vmin=-corr_vmax, vmax=corr_vmax)

    # ── HSS rows (sector fills, magma) ────────────────────────────────────
    for row, hk in enumerate(hss_keys):
        pct_label = hk.replace('hss_p', '') + 'th pct'
        for col, model_label in enumerate(model_labels):
            region_vals = {r['label']: metrics[model_label][r['label']][hk]
                           for r in REGIONS}
            draw_sector_panel(
                axs[row, col], region_vals,
                cmap='magma', norm=hss_norm, n_rad=n_rad,
                theta_ticks=theta_ticks, rad_ticks=[], rad_labels=[],
                row=row, col=col, title=model_label, row_label=pct_label
            )

    # ── NRMSE row (full 50x24, magma) ─────────────────────────────────────
    for col, model_label in enumerate(model_labels):
        draw_polar_panel(
            axs[n_hss, col], rmse_tables[model_label],
            cmap='magma_r', norm=nrmse_norm, n_rad=n_rad,
            theta_ticks=theta_ticks, rad_ticks=rad_ticks, rad_labels=rad_labels,
            row=n_hss, col=col, title=model_label, row_label='Norm RMSE'
        )

    # ── Corr row (full 50x24, bwr symmetric) ─────────────────────────────
    for col, model_label in enumerate(model_labels):
        draw_polar_panel(
            axs[n_hss + 1, col], corr_tables[model_label],
            cmap='bwr', norm=corr_norm, n_rad=n_rad,
            theta_ticks=theta_ticks, rad_ticks=rad_ticks, rad_labels=rad_labels,
            row=n_hss + 1, col=col, title=model_label, row_label='Corr'
        )

    # ── Three colourbars ──────────────────────────────────────────────────
    sm_hss = mpl.cm.ScalarMappable(norm=hss_norm, cmap='magma')
    sm_hss.set_array([])
    cbar_hss = fig.colorbar(sm_hss, ax=axs[:n_hss, :].ravel().tolist(),
                             orientation='vertical', fraction=0.03, pad=0.06, shrink=0.8)
    cbar_hss.set_label('HSS', fontsize=20, labelpad=10)
    cbar_hss.ax.tick_params(labelsize=20)

    sm_nrmse = mpl.cm.ScalarMappable(norm=nrmse_norm, cmap='magma_r')
    sm_nrmse.set_array([])
    cbar_nrmse = fig.colorbar(sm_nrmse, ax=axs[n_hss, :].ravel().tolist(),
                               orientation='vertical', fraction=0.007, pad=0.07, shrink=0.8)
    cbar_nrmse.set_label('Norm RMSE', fontsize=20, labelpad=10)
    cbar_nrmse.ax.tick_params(labelsize=20)

    sm_corr = mpl.cm.ScalarMappable(norm=corr_norm, cmap='bwr')
    sm_corr.set_array([])
    cbar_corr = fig.colorbar(sm_corr, ax=axs[n_hss + 1, :].ravel().tolist(),
                              orientation='vertical', fraction=0.007, pad=0.07, shrink=0.8)
    cbar_corr.set_label('Corr', fontsize=20, labelpad=10)
    cbar_corr.ax.tick_params(labelsize=20)

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    return fig

# ── Step 6: generate figures ──────────────────────────────────────────────
plot_combined_metrics_grid(
    metrics, rmse_tables, corr_tables,
    hss_keys=['hss_p50', 'hss_p75', 'hss_p90', 'hss_p99'],
    model_labels=list(MODEL_DICTS_ALL.keys()),
    hspace=-0.3, wspace=0.2, top=0.7, bottom=0.02, right=0.88,
    save_path=f'plots/model_results/regional_metrics_all_{RUN_TAG}.png',
    title='All'
)

plot_combined_metrics_grid(
    metrics, rmse_tables, corr_tables,
    hss_keys=['hss_p50', 'hss_p75', 'hss_p90', 'hss_p99'],
    model_labels=list(MODEL_DICTS_W.keys()),
    hspace=-0.3, wspace=0.2, top=0.7, bottom=0.02, right=0.88,
    save_path=f'plots/model_results/regional_metrics_weimer_{RUN_TAG}.png',
    title='Test Storm (May 2023)'
)

# plot_metric_polar(metrics, 'auc_pr', 'AUC-PR',
#                   cmap='magma', vmin=0, vmax=1,
#                   save_path=f'plots/model_results/regional_auc_pr_{RUN_TAG}.png')

print(f'\n{"Region":<22} {"p50":>8} {"p75":>8} {"p90":>8} {"p99":>8}')
print('-' * 58)
for region in REGIONS:
    t = universal_thresholds[region['label']]
    print(f'{region["label"]:<22} {t["p50"]:>8.4f} {t["p75"]:>8.4f} {t["p90"]:>8.4f} {t["p99"]:>8.4f}')
print()

## Regional time series

Per-region current through the storm with uncertainty bands.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Regional time series
# ═══════════════════════════════════════════════════════════════════════════
# Per-region current traced through the storm.
# ── Region time series plot ───────────────────────────────────────────────
def plot_region_time_series(
    results_dict, region_label, model_label='Model',
    window_minutes=HSS_WINDOW_MINUTES, figsize=(14, 4),
    thresholds=None, save_path=None
):
    """
    Plot the time series of mean |FAC| for AMPERE and predicted values
    for a named SHAP region, using non-overlapping block means matching
    the HSS calculation window.

    Parameters
    ----------
    results_dict  : dict       -- any of acorn_results, op_results, etc.
    region_label  : str        -- must match a 'label' in REGIONS
    model_label   : str        -- name shown in the legend for the predicted line
    window_minutes: int        -- block size in minutes (default: HSS_WINDOW_MINUTES)
    figsize       : tuple
    thresholds    : dict|None  -- {pkey: value} from universal_thresholds[region_label]
                                  if provided, HSS threshold lines are drawn at these
                                  values; if None, percentiles are computed locally
                                  from obs_block
    save_path     : str|None   -- if None, auto-generated from region_label and
                                  model_label
    """
    region = next((r for r in REGIONS if r['label'] == region_label), None)
    if region is None:
        raise ValueError(f"Region '{region_label}' not found. "
                         f"Valid labels: {[r['label'] for r in REGIONS]}")

    lat_idx, mlt_idx = region_idx(
        region['mlat_low'], region['mlat_high'],
        region['mlt_start'], region['mlt_end'],
    )
    if not lat_idx:
        raise ValueError(f"No valid lat indices for region '{region_label}' "
                         f"with LAT_MASK_50={LAT_MASK_50}")

    def _arr(v, field):
        x = v[field]
        return x.values if hasattr(x, 'values') else np.asarray(x)

    timestamps = sorted(results_dict.keys())
    obs_means, pred_means = [], []
    for ts in timestamps:
        v = results_dict[ts]
        obs_means.append(
            np.abs(_arr(v, 'ampere')   [np.ix_(lat_idx, mlt_idx)]).mean()
        )
        pred_means.append(
            np.abs(_arr(v, 'predicted')[np.ix_(lat_idx, mlt_idx)]).mean()
        )

    idx    = pd.to_datetime(timestamps)
    obs_s  = pd.Series(obs_means,  index=idx)
    pred_s = pd.Series(pred_means, index=idx)

    # Non-overlapping block means -- matching HSS calculation
    obs_block  = obs_s.resample(f'{window_minutes}min').mean().dropna()
    pred_block = pred_s.resample(f'{window_minutes}min').mean().dropna()
    n = min(len(obs_block), len(pred_block))
    obs_block  = obs_block.iloc[:n]
    pred_block = pred_block.iloc[:n]

    fig, ax = plt.subplots(figsize=figsize)

    # Raw 1-min values faint in background
    ax.plot(idx, obs_means,  color='steelblue', lw=0.5, alpha=0.2)
    ax.plot(idx, pred_means, color='orange',    lw=0.5, alpha=0.2)

    # Block means prominent
    ax.step(obs_block.index,  obs_block.values,  where='post',
            label=f'AMPERE ({window_minutes}-min block mean)',
            color='steelblue', lw=1.8)
    ax.step(pred_block.index, pred_block.values, where='post',
            label=f'{model_label} ({window_minutes}-min block mean)',
            color='orange', lw=1.8, alpha=0.9)

    # Percentile threshold lines -- use universal_thresholds if provided,
    # otherwise compute locally from obs_block
    if thresholds is not None:
        pct_lines = [
            (thresholds.get('p50', np.nan), '--', '50th pct'),
            (thresholds.get('p75', np.nan), '--', '75th pct'),
            (thresholds.get('p90', np.nan), '-.', '90th pct'),
            (thresholds.get('p99', np.nan), ':',  '99th pct'),
        ]
    else:
        pct_lines = [
            (np.percentile(obs_block.values, 50), '--', '50th pct'),
            (np.percentile(obs_block.values, 75), '--', '75th pct'),
            (np.percentile(obs_block.values, 90), '-.', '90th pct'),
            (np.percentile(obs_block.values, 99), ':',  '99th pct'),
        ]

    for thresh, ls, label in pct_lines:
        if np.isfinite(thresh):
            ax.axhline(thresh, color='steelblue', lw=1.0, linestyle=ls,
                       alpha=0.7, label=f'AMPERE {label} ({thresh:.3f})')

    ax.set_title(f'Mean |FAC| -- {region_label}  [{window_minutes}-min block mean]',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Time', fontsize=11)
    ax.set_ylabel(r'Mean |FAC| ($\mu$A/m²)', fontsize=11)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    plt.xticks(rotation=30, ha='right', fontsize=8)
    ax.margins(x=0)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=10)

    _save = save_path or (
        f'plots/model_results/region_ts_'
        f'{region_label.replace(" ", "_")}_'
        f'{model_label.replace(" ", "_")}_{RUN_TAG}.png'
    )
    os.makedirs(os.path.dirname(_save), exist_ok=True)
    plt.savefig(_save, dpi=150, bbox_inches='tight')
    plt.show()
    return fig, ax


# ── Example calls -- May 2023 storm ──────────────────────────────────────
may_acorn = {k: acorn_results[k] for k in may_keys if k in acorn_results}
may_bk_sci = {k: bk_sci_results[k] for k in may_keys if k in bk_sci_results}
may_op   = {k: op_results[k]   for k in may_keys if k in op_results}
may_bk_op   = {k: bk_op_results[k]   for k in may_keys if k in bk_op_results}
may_weimer   = {k: weimer_results[k]   for k in may_keys if k in weimer_results}

SECTOR = 'R0 Dayside'

fig, ax = plot_region_time_series(
    may_acorn, region_label=SECTOR, model_label='Sci',
    thresholds=universal_thresholds.get(SECTOR),
)

fig, ax = plot_region_time_series(
    may_bk_sci, region_label=SECTOR, model_label='BK Sci',
    thresholds=universal_thresholds.get(SECTOR),
)

fig, ax = plot_region_time_series(
    may_op, region_label=SECTOR, model_label='Opp',
    thresholds=universal_thresholds.get(SECTOR),
)

fig, ax = plot_region_time_series(
    may_bk_op, region_label=SECTOR, model_label='BK Opp',
    thresholds=universal_thresholds.get(SECTOR),
)

fig, ax = plot_region_time_series(
    may_weimer, region_label=SECTOR, model_label='weimer',
    thresholds=universal_thresholds.get(SECTOR),
)

## Conditional analysis by IMF BY_GSM

IMF BY controls the dawn-dusk asymmetry of the convection pattern. A model that
learned the physics reproduces that asymmetry reversing with BY; one that
learned climatology shows the same pattern in both splits.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Conditional analysis by IMF orientation
# ═══════════════════════════════════════════════════════════════════════════
# Splits the test set by upstream solar wind conditions, principally the IMF
# BY_GSM sign, which controls the dawn-dusk asymmetry of the convection
# pattern. A model that has learned the physics should reproduce that
# asymmetry reversing with BY; one that has learned climatology will show the
# same pattern in both splits.
split_dfs, split_names = splitting_omni_into_conditions(
    variable1='BZ_GSM', var_split=[-2,2], zero_split=True
)

split_key_sets = [
    {str(pd.Timestamp(ts)) for ts in df.index}
    for df in split_dfs
]

split_corr_tables, split_ampere_tables, split_pred_tables, split_unc_tables = [], [], [], []
op_split_corr,     op_split_pred     = [], []
opp_split_corr_w,   opp_split_pred_w   = [], []
sci_split_corr_w,   sci_split_pred_w   = [], []
bk_sci_split_corr,  bk_sci_split_pred  = [], []
bk_op_split_corr,  bk_op_split_pred  = [], []
bk_sci_split_corr_w, bk_sci_split_pred_w = [], []
bk_opp_split_corr_w, bk_opp_split_pred_w = [], []
weimer_split_corr, weimer_split_pred = [], []
mlt_bin_width = 1
lat_bin_width = 1

for key_set in split_key_sets:
    acorn_subset     = {k: v for k, v in acorn_results.items()   if k in key_set}
    opp_subset       = {k: v for k, v in op_results.items()     if k in key_set}
    bk_sci_subset    = {k: v for k, v in bk_sci_results.items()  if k in key_set}
    bk_opp_subset    = {k: v for k, v in bk_op_results.items()  if k in key_set}
    weimer_subset    = {k: v for k, v in weimer_results.items()   if k in key_set}
    acorn_subset_w   = {k: v for k, v in acorn_subset.items()   if k in weimer_results}
    opp_subset_w     = {k: v for k, v in opp_subset.items()     if k in weimer_results}
    bk_sci_subset_w  = {k: v for k, v in bk_sci_subset.items()  if k in weimer_results}
    bk_opp_subset_w  = {k: v for k, v in bk_opp_subset.items()  if k in weimer_results}

    split_corr_tables.append(creating_correlation_tables(
        acorn_subset, mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    split_ampere_tables.append(creating_mean_tables(
        acorn_subset, value='ampere', mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    split_pred_tables.append(creating_mean_tables(
        acorn_subset, value='predicted', mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    split_unc_tables.append(creating_mean_tables(
        acorn_subset, value='std', mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))

    op_split_corr.append(creating_correlation_tables(
        opp_subset, mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    op_split_pred.append(creating_mean_tables(
        opp_subset, value='predicted', mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))

    bk_sci_split_corr.append(creating_correlation_tables(
        bk_sci_subset, mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    bk_sci_split_pred.append(creating_mean_tables(
        bk_sci_subset, value='predicted', mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    bk_op_split_corr.append(creating_correlation_tables(
        bk_opp_subset, mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    bk_op_split_pred.append(creating_mean_tables(
        bk_opp_subset, value='predicted', mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))

    # # Weimer-window-only variants
    # sci_split_corr_w.append(creating_correlation_tables(
    #     acorn_subset_w, mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    # sci_split_pred_w.append(creating_mean_tables(
    #     acorn_subset_w, value='predicted', mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    # opp_split_corr_w.append(creating_correlation_tables(
    #     opp_subset_w, mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    # opp_split_pred_w.append(creating_mean_tables(
    #     opp_subset_w, value='predicted', mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    # bk_sci_split_corr_w.append(creating_correlation_tables(
    #     bk_sci_subset_w, mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    # bk_sci_split_pred_w.append(creating_mean_tables(
    #     bk_sci_subset_w, value='predicted', mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    # bk_opp_split_corr_w.append(creating_correlation_tables(
    #     bk_opp_subset_w, mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    # bk_opp_split_pred_w.append(creating_mean_tables(
    #     bk_opp_subset_w, value='predicted', mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))

    # weimer_split_corr.append(creating_correlation_tables(
    #     weimer_subset, mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))
    # weimer_split_pred.append(creating_mean_tables(
    #     weimer_subset, value='predicted', mlt_bin_width=mlt_bin_width, lat_bin_width=lat_bin_width))


## Conditional polar plots

In [ ]:
theta_ticks = np.linspace(0, 2*np.pi, 8, endpoint=False)
n_rad      = 40 if LAT_MASK_50 else 50
rad_div     = 5 if LAT_MASK_50 else 5
rad_step   = max(1, n_rad // 5)
rad_ticks   = list(range(0, n_rad, rad_step))
rad_labels  = ['']+[str(int(80 - t)) for t in rad_ticks[:-1]]


def plot_conditional_polar(row_specs, split_names, suptitle, save_path=None,
                             polar_ylim=None):
    # Filter to mean (bwr) rows only
    row_specs = [(tables, cmap, cbar_label, row_label)
                 for tables, cmap, cbar_label, row_label in row_specs
                 if cmap == 'bwr']

    n_rows = len(row_specs)
    n_cols = len(split_names)
    fig, axs = plt.subplots(
        nrows=n_rows, ncols=n_cols,
        figsize=(5 * n_cols, 5 * n_rows),
        subplot_kw=dict(projection='polar')
    )
    if n_rows == 1:
        axs = axs[np.newaxis, :]

    # Shared bwr scale across all rows
    all_vals = [
        np.abs(df).max().max()
        for tables, _, _, _ in row_specs
        for df in tables
        if not df.empty
    ]
    scale = max(all_vals) if all_vals else 1.0
    norm  = mpl.colors.Normalize(vmin=-scale, vmax=scale)

    _ylim      = polar_ylim if polar_ylim is not None else n_rad
    rad_ticks  = [t for t in range(0, _ylim, rad_step) if (90 - t) % 10 == 0]
    rad_labels = ['']+[str(int(80 - t)) for t in rad_ticks[:-1]]

    for j, (tables, cmap, cbar_label, row_label) in enumerate(row_specs):
        if all(df.empty for df in tables):
            for ax in axs[j]:
                ax.set_visible(False)
            continue

        ref_df     = next(df for df in tables if not df.empty)
        n_mlt_b    = len(ref_df)
        n_lat_plot = _ylim

        r_row, th_row = np.meshgrid(
            np.linspace(0, _ylim, n_lat_plot, endpoint=False),
            np.linspace(0, 2*np.pi, n_mlt_b, endpoint=False)
        )

        for i, (df, col_name) in enumerate(zip(tables, split_names)):
            ax = axs[j, i]
            if i == 0:
                ax.set_ylabel(row_label, fontsize=19)
            if j == 0:
                ax.set_title(col_name, fontsize=19)
            if df.empty:
                ax.set_visible(False)
                continue
            c = ax.pcolormesh(th_row, r_row, df.iloc[:, :n_lat_plot], cmap=cmap, norm=norm)
            polar_setup(ax, n_rad, rad_ticks=rad_ticks, rad_labels=rad_labels,
                        grid_alpha=0.5, ylim=_ylim)

    # Single shared colorbar for all rows
    fig.colorbar(
        mpl.cm.ScalarMappable(norm=norm, cmap='bwr'),
        ax=axs.ravel().tolist(),
        label=r'Mean ($\mu$A/m²)',
        orientation='vertical',
        pad=0.1,
        shrink=0.7
    )
    fig.subplots_adjust(hspace=-0.45, wspace=0.1, top=0.9, bottom=0.05, right=0.77)
    plt.suptitle(suptitle, fontsize=25, fontweight='bold', y=0.9)
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150)
    plt.show()

split_names = [r'$B^{IMF}_z$ <= -2nT', r'-2nT < $B^{IMF}_z$ <= 0nT', r'0nT < $B^{IMF}_z$ <= 2nT', r'$B^{IMF}_z$ > 2nT']
# split_names = [r'$B^{IMF}_z$ <= -2nT', r'-2nT < $B^{IMF}_z$ <= 0nT', r'0nT < $B^{IMF}_z$ <= 2nT']
# ── Figure A: Full test set ───────────────────────────────────────────────
plot_conditional_polar([
    (split_ampere_tables, 'bwr',     r'Mean ($\mu$A/$m^2$)', 'Ampere'),
    (split_pred_tables,   'bwr',     r'Mean ($\mu$A/$m^2$)', r'ACORN sci'),
    (op_split_pred,      'bwr',     r'Mean ($\mu$A/$m^2$)', r'ACORN Op'),
    # (bk_sci_split_pred,   'bwr',     r'Mean ($\mu$A/$m^2$)', r'BK sci'),
    # (bk_op_split_pred,   'bwr',     r'Mean ($\mu$A/$m^2$)', r'BK Op'),
    # (split_unc_tables,    'Purples', r'Mean ($\mu$A/$m^2$)', r'ACORN sci $\sigma$'),
    # (split_corr_tables,   'viridis', 'Corr Coeff',            'ACORN sci Corr'),
    # (op_split_corr,      'viridis', 'Corr Coeff',            'ACORN Op Corr'),
    # (bk_sci_split_corr,   'viridis', 'Corr Coeff',            'BK sci Corr'),
    # (bk_op_split_corr,   'viridis', 'Corr Coeff',            'BK Op Corr'),
], split_names, r'$B^{IMF}_z$ Conditional Averages — Full Test Set',
   f'plots/model_results/conditional_all_Bz.png', polar_ylim=15)



In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MLT-slice integrated current and profile comparison
# ═══════════════════════════════════════════════════════════════════════════
# Integrates along a single MLT meridian rather than over the whole cap,
# giving a latitude profile through the current sheets at one local time.
#
# This is where ACORN's spatial smoothing is most visible: the predicted
# profile has the right integral and the peaks in roughly the right place,
# but the peaks are lower and broader than observed. That follows from
# training against a conditional mean, which averages over the plausible
# ionospheric states consistent with a given solar wind input.
def calculate_integrated_currents_mlt(data_col, mlt_value, radius=6371200):
    """
    Compute integrated current for a single MLT column slice.
    grid_surface_integral requires at least 2 columns, so we duplicate
    the column and halve the result to get the single-column integral.
    """
    _n_lats   = len(data_col)
    _lat_vals = 90 - np.arange(1, _n_lats + 1)

    mlat_grid   = np.tile(_lat_vals.reshape(-1, 1), (1, 2))
    mlt_grid    = np.tile(np.array([float(mlt_value), float(mlt_value) + 1]), (_n_lats, 1))
    col         = data_col.copy()
    col[np.abs(col) <= 0.1] = 0
    grid_values = np.tile(np.abs(col).reshape(-1, 1), (1, 2))

    return grid_surface_integral(
        grid_lats=mlat_grid,
        grid_azis=mlt_grid,
        grid_values=grid_values,
        sphere_radius=radius,
        aziunit='hour'
    ) / 2

def plot_polar_with_mlt_line(ax, grid_50x24, mlt_value, mlat_axis, title, vmax):
    n_lats, n_mlts = grid_50x24.shape
    theta_edges = np.linspace(0, 2 * np.pi, n_mlts + 1)
    lat_edges   = np.linspace(0, n_lats, n_lats + 1)
    th_mesh, r_mesh = np.meshgrid(theta_edges, lat_edges)

    pcm = ax.pcolormesh(th_mesh, r_mesh, grid_50x24, cmap="bwr",
                        vmin=-vmax, vmax=vmax, shading="flat")

    # This figure labels every 3-hour MLT tick rather than alternating.
    mlt_ticks = np.arange(0, n_mlts, 3)
    polar_setup(ax, n_lats, mlt_labels=[str(t) for t in mlt_ticks],
                mlt_fontsize=12, ylim=n_lats)

    lat_tick_rows = np.arange(0, n_lats, 10)
    ax.set_yticks(lat_tick_rows)
    ax.set_yticklabels([f"{mlat_axis[r]+1:.0f}" + chr(176)
                        for r in lat_tick_rows], fontsize=12)

    theta_line = (mlt_value + 0.5) / n_mlts * 2 * np.pi
    ax.plot([theta_line, theta_line], [0, n_lats], color="lime", lw=2.5, zorder=5)

    ax.set_title(title, fontsize=20)
    return pcm


def find_extrema_with_min_separation(profile, mlat_axis, min_separation_deg):
    peak_idx, _   = find_peaks(profile)
    trough_idx, _ = find_peaks(-profile)

    candidates = (
        [{"index": i, "sign":  1} for i in peak_idx] +
        [{"index": i, "sign": -1} for i in trough_idx]
    )
    for c in candidates:
        c["position"]  = mlat_axis[c["index"]]
        c["amplitude"] = profile[c["index"]]

    candidates.sort(key=lambda c: abs(c["amplitude"]), reverse=True)

    kept = []
    for c in candidates:
        if all(abs(c["position"] - k["position"]) >= min_separation_deg
               for k in kept):
            kept.append(c)

    peaks_kept   = sorted([k["index"] for k in kept if k["sign"] ==  1])
    troughs_kept = sorted([k["index"] for k in kept if k["sign"] == -1])
    return peaks_kept, troughs_kept


def plot_raw_peaks(profile, mlat_axis, min_separation_deg, ax, title,
                   color='k', label=None):
    peaks, troughs = find_extrema_with_min_separation(
        profile, mlat_axis, min_separation_deg)

    ax.plot(mlat_axis, profile, color=color, lw=1.2,
            label=label if label else title)
    ax.axhline(0, color="gray", lw=0.6)

    for i in peaks:
        ax.annotate(f"{mlat_axis[i]:.0f}" + chr(176),
                    (mlat_axis[i], profile[i]),
                    textcoords="offset points", xytext=(0, 8),
                    ha="center", fontsize=10, color=color,
                    arrowprops=dict(arrowstyle='->', color=color, lw=0.8))
    for i in troughs:
        ax.annotate(f"{mlat_axis[i]:.0f}" + chr(176),
                    (mlat_axis[i], profile[i]),
                    textcoords="offset points", xytext=(0, -12),
                    ha="center", fontsize=10, color=color,
                    arrowprops=dict(arrowstyle='->', color=color, lw=0.8))

    ax.set_xlim(mlat_axis.max(), mlat_axis.min())
    ax.set_xlabel("MLAT (deg)", fontsize=15)
    ax.set_ylabel(r'FAC ($\mu$A/m²)', fontsize=15)
    if title:
        ax.set_title(title, fontsize=10)
    if label:
        ax.legend(fontsize=7)


def build_timestamp_figure(key):
    ampere_grid = acorn_results[key]['ampere'].to_numpy()
    pred_grid   = acorn_results[key]['predicted'].to_numpy()
    opp_grid    = op_results[key]['predicted'].to_numpy() \
                  if key in op_results else np.zeros_like(pred_grid)
    vmax        = max(np.nanmax(np.abs(ampere_grid)),
                      np.nanmax(np.abs(pred_grid)),
                      np.nanmax(np.abs(opp_grid)))

    # ── Integrated current at selected MLT ───────────────────────────────
    amp_int  = calculate_integrated_currents_mlt(ampere_grid[:, DEMO_MLT], DEMO_MLT)
    sci_int  = calculate_integrated_currents_mlt(pred_grid[:,  DEMO_MLT],  DEMO_MLT)
    opp_int  = calculate_integrated_currents_mlt(opp_grid[:,   DEMO_MLT],  DEMO_MLT)
    amp_int_label = f'Integrated J: {amp_int/1e9:.2f} kA'
    sci_int_label = f'Integrated J: {sci_int/1e9:.2f} kA'
    opp_int_label = f'Integrated J: {opp_int/1e9:.2f} kA'

    fig = plt.figure(figsize=(18, 11))
    gs  = fig.add_gridspec(2, 3, height_ratios=[1.3, 1],
                            hspace=0.15, wspace=0.2)

    ax_polar_amp  = fig.add_subplot(gs[0, 0], projection="polar")
    ax_polar_sci  = fig.add_subplot(gs[0, 1], projection="polar")
    ax_polar_opp  = fig.add_subplot(gs[0, 2], projection="polar")
    ax_compare    = fig.add_subplot(gs[1, :])

    # ── Polar maps ────────────────────────────────────────────────────────
    pcm0 = plot_polar_with_mlt_line(
        ax_polar_amp, ampere_grid, DEMO_MLT, mlat_axis,
        f'AMPERE', vmax)
    pcm1 = plot_polar_with_mlt_line(
        ax_polar_sci, pred_grid, DEMO_MLT, mlat_axis,
        f'ACORN Sci', vmax)
    pcm2 = plot_polar_with_mlt_line(
        ax_polar_opp, opp_grid, DEMO_MLT, mlat_axis,
        f'ACORN Op', vmax)

    cbar_ax = fig.add_axes([0.92, 0.47, 0.015, 0.4])  # [left, bottom, width, height]
    cbar = fig.colorbar(
        mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=-vmax, vmax=vmax), cmap='bwr'),
        cax=cbar_ax,
        label=r'FAC ($\mu$A/m²)',
    )
    cbar.set_label(r'FAC ($\mu$A/m²)', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=12)

    # ── Overlay comparison panel ──────────────────────────────────────────
    plot_raw_peaks(ampere_grid[:, DEMO_MLT], mlat_axis, MIN_SEPARATION_DEG,
                   ax_compare, '', color='black',
                   label=f'AMPERE  ({amp_int_label})')
    plot_raw_peaks(pred_grid[:,  DEMO_MLT],  mlat_axis, MIN_SEPARATION_DEG,
                   ax_compare, '', color='orange',
                   label=f'ACORN Sci ({sci_int_label})')
    plot_raw_peaks(opp_grid[:,   DEMO_MLT],  mlat_axis, MIN_SEPARATION_DEG,
                   ax_compare, '', color='green',
                   label=f'ACORN Op ({opp_int_label})')
    ax_compare.set_title('AMPERE vs ACORN Sci vs ACORN Op — Direct Comparison',
                          fontsize=17)
    raw_peak_max = max(np.nanmax(np.abs(ampere_grid[:,DEMO_MLT])),
                      np.nanmax(np.abs(pred_grid[:,DEMO_MLT])),
                      np.nanmax(np.abs(opp_grid[:,DEMO_MLT])))
    raw_peak_ylim = raw_peak_max+raw_peak_max*0.3
    ax_compare.set_ylim(-raw_peak_ylim, raw_peak_ylim)
    ax_compare.legend(fontsize=15)

    fig.suptitle(f'{key}  —  MLT = {DEMO_MLT}h', fontsize=30, fontweight='bold')
    _fname = (f'plots/model_results/detailed_current_wedge_'
              f'{key.replace(":", ";").replace(" ", "_")}_mlt{DEMO_MLT}_{RUN_TAG}.png')
    os.makedirs(os.path.dirname(_fname), exist_ok=True)
    plt.savefig(_fname, dpi=150)
    plt.show()


# ── Config ────────────────────────────────────────────────────────────────
DEMO_MLT           = 19
MIN_SEPARATION_DEG = 2.0
RANDOM_SEED        = 1
N_TIMES            = 2
MANUAL_TIMESTAMPS  = ['2023-05-06 04:00:00']
# MANUAL_TIMESTAMPS = []

mlat_axis   = ACORN_MLAT_VALS
all_keys   = sorted(acorn_results.keys())
valid_keys = [k for k in all_keys
               if np.any(np.abs(acorn_results[k]['ampere'].to_numpy()) > 0)]

if MANUAL_TIMESTAMPS:
    demo_keys = []
    for ts in MANUAL_TIMESTAMPS:
        candidate  = ts if isinstance(ts, str) else str(ts)
        normalized = pd.Timestamp(candidate).strftime('%Y-%m-%d %H:%M:%S')
        if candidate in acorn_results:
            demo_keys.append(candidate)
        elif normalized in acorn_results:
            demo_keys.append(normalized)
        else:
            print(f"Warning: '{ts}' not found in acorn_results -- skipping")
    if not demo_keys:
        raise ValueError("None of MANUAL_TIMESTAMPS were found in acorn_results")
else:
    rng       = np.random.default_rng(RANDOM_SEED)
    demo_keys = list(rng.choice(valid_keys, size=N_TIMES, replace=False))

print("Selected timestamps:", demo_keys)
for key in demo_keys:
    build_timestamp_figure(key)